# Result Availability Audit — Part 1 of 3
## Latency measurement, cohort construction, and visibility characterisation

**Study construct.** Every laboratory result carries two clinically distinct times: the time the
observation was made (specimen collection / measurement) and the time the result was entered into
the record and became visible to staff. Prediction models are almost universally built on the first.
Deployed decision support can only act on the second. This notebook measures the interval between
them and characterises how much of a patient's laboratory record is actually visible at any given
scoring time.

**Part 1 (this notebook) produces:**

| Artefact | Content |
|---|---|
| `cohort_mimic.parquet` | MIMIC-IV analytic ICU cohort with admission metadata |
| `cohort_eicu.parquet` | eICU-CRD analytic ICU cohort with hospital metadata |
| `labs_mimic.parquet` | Row-level labs, both clocks, latency, priority, harmonised analyte |
| `labs_eicu.parquet` | Row-level labs, both clocks, latency, harmonised analyte |
| `micro_mimic.parquet` | Microbiology, both clocks, latency |
| `t1_*.csv` / `f1_*.png` | Latency descriptive tables and figures |
| `visibility_curve.csv` | Proportion of drawn results visible at each scoring hour |
| `part1_manifest.json` | Hash-chained provenance record of every input and output |

**Part 2 will consume these** to build the two-clock hourly feature matrices, define escalation
events, and score models under each clock.
**Part 3 will produce** lead time, delayed and missed detection counts, the comparator arm, and the
final figures.

**Design decisions that matter for the analysis**

1. The latency variable is *validated before it is used*. Negative, zero, and implausibly long
   intervals are quantified explicitly rather than silently dropped, because a timestamp field is
   only usable if its failure modes are known.
2. Point-of-care and blood gas analytes act as an internal fast-turnaround control. If the measure
   is behaving, these post far faster than send-out chemistry within the same database.
3. eICU is treated as a **secondary** latency site. Its second timestamp is documented as the time a
   *revised* value was entered, which is a proxy for first availability rather than the thing itself.
   The notebook quantifies how often the two offsets coincide so this assumption is testable rather
   than assumed.
4. Analyte identifiers are resolved by matching against each database's own dictionary and the
   resolved mapping is printed for inspection. Hard-coded identifier lists are not trusted.

**Expected runtime.** First run 10–25 minutes, dominated by the single pass over `labevents.csv`
(~13 GB) and `lab.csv` (~5 GB). Every heavy step is cached to Parquet and skipped on re-run.

---
## 1. Environment

In [1]:
import sys, os, json, time, hashlib, platform, warnings, re, gc
from pathlib import Path
from datetime import datetime, timezone
from dataclasses import dataclass, field, asdict

warnings.filterwarnings("ignore")

REQUIRED = {"pandas": "pandas", "numpy": "numpy", "pyarrow": "pyarrow", "matplotlib": "matplotlib"}
OPTIONAL = {"duckdb": "duckdb"}

missing = []
for mod, pkg in REQUIRED.items():
    try:
        __import__(mod)
    except ImportError:
        missing.append(pkg)
if missing:
    raise SystemExit(
        "Missing required packages: " + ", ".join(missing)
        + "\nInstall with:  pip install " + " ".join(missing)
    )

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

HAVE_DUCKDB = False
try:
    import duckdb
    HAVE_DUCKDB = True
except ImportError:
    pass

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)
pd.set_option("display.max_rows", 200)
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")

ENV_VERSIONS = {
    "python": platform.python_version(),
    "platform": f"{platform.system()} {platform.release()}",
    "pandas": pd.__version__,
    "numpy": np.__version__,
    "pyarrow": pa.__version__,
    "duckdb": duckdb.__version__ if HAVE_DUCKDB else None,
}

print(f"python      {platform.python_version()}  ({platform.system()} {platform.release()})")
print(f"pandas      {pd.__version__}")
print(f"numpy       {np.__version__}")
print(f"pyarrow     {pa.__version__}")
print(f"duckdb      {duckdb.__version__ if HAVE_DUCKDB else 'NOT INSTALLED - falling back to chunked pandas (much slower)'}")
if not HAVE_DUCKDB:
    print("\n  Strongly recommended:  pip install duckdb")
    print("  It reduces the labevents pass from roughly 20 minutes to roughly 3.")

python      3.11.5  (Windows 10)
pandas      2.3.3
numpy       1.24.4
pyarrow     24.0.0
duckdb      NOT INSTALLED - falling back to chunked pandas (much slower)

  Strongly recommended:  pip install duckdb
  It reduces the labevents pass from roughly 20 minutes to roughly 3.


---
## 2. Configuration

Everything the analysis depends on lives here. The configuration is hashed into the run manifest so
that any result can be traced to the exact parameter set that produced it.

In [2]:
@dataclass(frozen=True)
class Config:
    # ---- source locations -------------------------------------------------
    mimic_root: str = r"C:\mimic-iv-2.2"
    eicu_root:  str = r"C:\Users\kruta\Downloads\eicu-collaborative-research-database-2.0"
    out_root:   str = r"C:\Research_Paper_2\result_availability_audit"

    # ---- cohort definition ------------------------------------------------
    min_age_years: int = 18
    max_age_years: int = 120
    min_icu_hours: float = 12.0     # stay must be long enough to host a scoring window
    max_icu_hours: float = 24 * 30  # guard against pathological records
    first_stay_only: bool = True    # one ICU stay per patient, avoids within-patient clustering

    # ---- lab window relative to ICU admission -----------------------------
    pre_icu_hours: float = 6.0      # keep pre-ICU labs, flagged separately
    analysis_horizon_hours: float = 72.0

    # ---- latency validity rules -------------------------------------------
    latency_min_minutes: float = 0.0        # below this is a timestamp inconsistency
    latency_max_minutes: float = 60 * 24    # above this is treated as an outlier, quantified not hidden
    micro_latency_max_minutes: float = 60 * 24 * 14  # cultures legitimately take days

    # ---- visibility curve -------------------------------------------------
    visibility_hours: tuple = tuple(range(1, 49))

    # ---- engine -----------------------------------------------------------
    duckdb_threads: int = 0             # 0 -> all logical cores
    duckdb_memory_limit_gb: int = 6
    pandas_chunk_rows: int = 3_000_000  # fallback path only
    full_hash_max_gb: float = 2.0       # files larger than this get a sampled fingerprint

    # ---- reproducibility --------------------------------------------------
    seed: int = 20260906
    force_rebuild: bool = False         # True re-runs every cached stage

CFG = Config()

MIMIC = Path(CFG.mimic_root)
EICU  = Path(CFG.eicu_root)
OUT   = Path(CFG.out_root)

DIR = {
    "cache":  OUT / "cache",
    "tables": OUT / "tables",
    "figs":   OUT / "figures",
    "logs":   OUT / "logs",
}
for d in DIR.values():
    d.mkdir(parents=True, exist_ok=True)

np.random.seed(CFG.seed)

CFG_JSON = json.dumps({k: (list(v) if isinstance(v, tuple) else v)
                       for k, v in asdict(CFG).items()}, sort_keys=True)
CFG_HASH = hashlib.sha256(CFG_JSON.encode()).hexdigest()

print("MIMIC-IV root :", MIMIC, "" if MIMIC.exists() else "  <-- NOT FOUND")
print("eICU root     :", EICU,  "" if EICU.exists()  else "  <-- NOT FOUND")
print("Output root   :", OUT)
print("Config hash   :", CFG_HASH[:16])

MIMIC-IV root : C:\mimic-iv-2.2 
eICU root     : C:\Users\kruta\Downloads\eicu-collaborative-research-database-2.0 
Output root   : C:\Research_Paper_2\result_availability_audit
Config hash   : 27865ff45285c118


---
## 3. Reproducibility scaffold

Each artefact written is hashed, and each hash is chained into the previous one. A single terminal
digest therefore fixes the entire input and output set of the run. This is what makes a later claim
that "Table 2 came from this exact extraction" checkable rather than asserted.

In [3]:
class Provenance:
    """SHA-256 hash chain over inputs and outputs, plus per-stage timing."""

    def __init__(self, seed_material: str):
        self.chain = hashlib.sha256(seed_material.encode()).hexdigest()
        self.records = []
        self.timings = {}

    @staticmethod
    def file_digest(path: Path, full_hash_max_bytes: int):
        """Full SHA-256 for modest files; a documented sampled fingerprint for very large ones."""
        size = path.stat().st_size
        h = hashlib.sha256()
        if size <= full_hash_max_bytes:
            with open(path, "rb") as f:
                for block in iter(lambda: f.read(8 << 20), b""):
                    h.update(block)
            return h.hexdigest(), "sha256_full", size
        # sampled fingerprint: size + head + tail + three interior probes
        h.update(str(size).encode())
        probe = 16 << 20
        offsets = [0, size // 4, size // 2, (3 * size) // 4, max(0, size - probe)]
        with open(path, "rb") as f:
            for off in offsets:
                f.seek(off)
                h.update(f.read(probe))
        return h.hexdigest(), "sha256_sampled", size

    def add(self, role: str, name: str, path: Path, extra=None):
        path = Path(path)
        if not path.exists():
            digest, method, size = "MISSING", "none", 0
        else:
            digest, method, size = self.file_digest(path, int(CFG.full_hash_max_gb * (1 << 30)))
        rec = {
            "role": role, "name": name, "path": str(path),
            "bytes": size, "digest": digest, "digest_method": method,
            "recorded_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        }
        if extra:
            rec.update(extra)
        self.chain = hashlib.sha256((self.chain + digest).encode()).hexdigest()
        rec["chain_after"] = self.chain
        self.records.append(rec)
        return rec

    def time(self, stage: str, seconds: float, detail=None):
        self.timings[stage] = {"seconds": round(seconds, 2), "detail": detail or {}}

    def manifest(self):
        return {
            "generated_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
            "config": json.loads(CFG_JSON),
            "config_sha256": CFG_HASH,
            "environment": {**ENV_VERSIONS, "cpu_count": os.cpu_count()},
            "records": self.records,
            "timings": self.timings,
            "terminal_chain_sha256": self.chain,
        }


PROV = Provenance(CFG_HASH)


class Stage:
    """Context manager that times a stage and reports it."""
    def __init__(self, label):
        self.label = label
    def __enter__(self):
        self.t0 = time.time()
        print(f"[{self.label}] start")
        return self
    def __exit__(self, *exc):
        dt = time.time() - self.t0
        PROV.time(self.label, dt)
        print(f"[{self.label}] done in {dt/60:.2f} min" if dt > 90 else f"[{self.label}] done in {dt:.1f} s")
        return False


def cached(path: Path):
    """True when a cached artefact may be reused."""
    return path.exists() and not CFG.force_rebuild


def write_table(df: pd.DataFrame, name: str, index=False):
    p = DIR["tables"] / f"{name}.csv"
    df.to_csv(p, index=index)
    PROV.add("table", name, p, {"rows": int(len(df)), "cols": int(df.shape[1])})
    return p


def write_parquet(df: pd.DataFrame, name: str):
    p = DIR["cache"] / f"{name}.parquet"
    df.to_parquet(p, index=False, compression="zstd")
    PROV.add("cache", name, p, {"rows": int(len(df)), "cols": int(df.shape[1])})
    return p


def save_fig(fig, name: str):
    p = DIR["figs"] / f"{name}.png"
    fig.savefig(p, dpi=200, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    PROV.add("figure", name, p)
    return p

print("Provenance chain seeded:", PROV.chain[:16])

Provenance chain seeded: cad52f1e7cac4d99


---
## 4. Source file resolution

Both downloads are laid out with each CSV nested inside a directory of the same name
(`hosp/labevents.csv/labevents.csv`). The resolver handles that, the flat layout, and gzipped
variants, so the notebook runs unchanged if the directory structure is later normalised.

In [4]:
def resolve_file(root: Path, module: str, filename: str) -> Path:
    """Locate a source CSV across the nested, flat, and gzipped layouts."""
    base = root / module if module else root
    stem = filename[:-4] if filename.endswith(".csv") else filename
    candidates = [
        base / filename,                        # flat:   hosp/labevents.csv
        base / filename / filename,             # nested: hosp/labevents.csv/labevents.csv
        base / f"{filename}.gz",
        base / filename / f"{filename}.gz",
        base / stem / filename,
        base / stem / f"{filename}.gz",
    ]
    for c in candidates:
        if c.is_file():
            return c
    # last resort: shallow glob
    if base.exists():
        for pat in (f"{stem}.csv", f"{stem}.csv.gz"):
            hits = sorted(base.glob(f"**/{pat}"))
            if hits:
                return hits[0]
    raise FileNotFoundError(
        f"Could not locate '{filename}' under {base}\nTried:\n  " + "\n  ".join(str(c) for c in candidates)
    )


MIMIC_FILES = {
    "icustays":            ("icu",  "icustays.csv"),
    "patients":            ("hosp", "patients.csv"),
    "admissions":          ("hosp", "admissions.csv"),
    "d_labitems":          ("hosp", "d_labitems.csv"),
    "labevents":           ("hosp", "labevents.csv"),
    "microbiologyevents":  ("hosp", "microbiologyevents.csv"),
}

EICU_FILES = {
    "patient":  ("", "patient.csv"),
    "hospital": ("", "hospital.csv"),
    "lab":      ("", "lab.csv"),
}

SRC = {}
problems = []

for key, (mod, fn) in MIMIC_FILES.items():
    try:
        SRC[f"mimic.{key}"] = resolve_file(MIMIC, mod, fn)
    except FileNotFoundError as e:
        problems.append(str(e))

for key, (mod, fn) in EICU_FILES.items():
    try:
        SRC[f"eicu.{key}"] = resolve_file(EICU, mod, fn)
    except FileNotFoundError as e:
        problems.append(str(e))

if problems:
    print("UNRESOLVED SOURCES\n" + "\n\n".join(problems) + "\n")

inv = pd.DataFrame(
    [{"source": k, "gb": round(p.stat().st_size / (1 << 30), 3), "path": str(p)}
     for k, p in SRC.items()]
).sort_values("gb", ascending=False).reset_index(drop=True)

display(inv)
print(f"\nTotal source volume: {inv['gb'].sum():.2f} GB across {len(inv)} files")

,source,gb,path
0,mimic.labevents,12.787,C:\mimic-iv-2.2\hosp\labevents.csv\labevents.csv
1,eicu.lab,2.202,C:\Users\kruta\Downloads\eicu-collaborative-re...
2,mimic.microbiologyevents,0.690,C:\mimic-iv-2.2\hosp\microbiologyevents.csv\mi...
3,mimic.admissions,0.068,C:\mimic-iv-2.2\hosp\admissions.csv\admissions...
4,eicu.patient,0.045,C:\Users\kruta\Downloads\eicu-collaborative-re...
5,mimic.icustays,0.011,C:\mimic-iv-2.2\icu\icustays.csv\icustays.csv
6,mimic.patients,0.009,C:\mimic-iv-2.2\hosp\patients.csv\patients.csv
7,mimic.d_labitems,0.000,C:\mimic-iv-2.2\hosp\d_labitems.csv\d_labitems...
8,eicu.hospital,0.000,C:\Users\kruta\Downloads\eicu-collaborative-re...



Total source volume: 15.81 GB across 9 files


In [5]:
# Integrity fingerprints. Large files use the documented sampled method so this stays fast.
with Stage("source-fingerprints"):
    for k, p in SRC.items():
        PROV.add("source", k, p)

fp = pd.DataFrame(PROV.records)[["name", "bytes", "digest_method", "digest"]].copy()
fp["gb"] = (fp["bytes"] / (1 << 30)).round(3)
fp["digest"] = fp["digest"].str.slice(0, 16)
display(fp[["name", "gb", "digest_method", "digest"]])

[source-fingerprints] start
[source-fingerprints] done in 2.0 s


,name,gb,digest_method,digest
0,mimic.icustays,0.011,sha256_full,5d7ec6eeedf1b030
1,mimic.patients,0.009,sha256_full,d6920a7f4efad739
2,mimic.admissions,0.068,sha256_full,59a99ad5a5549f1f
3,mimic.d_labitems,0.000,sha256_full,55106120ac39d031
4,mimic.labevents,12.787,sha256_sampled,93f88984eb8554c1
5,mimic.microbiologyevents,0.690,sha256_full,2e5b13bfee8e81a7
6,eicu.patient,0.045,sha256_full,cc022f6bb49ae477
7,eicu.hospital,0.000,sha256_full,69ec9aa224122bd4
8,eicu.lab,2.202,sha256_sampled,bb8bc1e073552ac7


---
## 5. Query engine

DuckDB streams the large CSVs out of core and pushes the column and row filters into the scan, which
is the difference between a three minute pass over `labevents.csv` and a twenty minute one. A chunked
pandas path is provided so the notebook still completes without it.

In [6]:
def sqlpath(p) -> str:
    """DuckDB string literal for a Windows path."""
    return str(Path(p).as_posix()).replace("'", "''")


CON = None
if HAVE_DUCKDB:
    CON = duckdb.connect(database=":memory:")
    threads = CFG.duckdb_threads or (os.cpu_count() or 4)
    CON.execute(f"PRAGMA threads={threads}")
    CON.execute(f"PRAGMA memory_limit='{CFG.duckdb_memory_limit_gb}GB'")
    CON.execute(f"PRAGMA temp_directory='{sqlpath(DIR['cache'] / '_spill')}'")
    print(f"DuckDB ready: {threads} threads, {CFG.duckdb_memory_limit_gb} GB memory limit, spill to disk enabled")


def dsql(query: str) -> pd.DataFrame:
    if CON is None:
        raise RuntimeError("DuckDB not available for this operation")
    return CON.execute(query).df()


def read_csv_expr(path, **kw) -> str:
    """
    Build a read_csv_auto() expression. Kept version-tolerant: no `types=` map, and every
    downstream cast uses TRY_CAST so inference differences cannot break the pipeline.
    """
    opts = ["header=true", "sample_size=1048576", "ignore_errors=false"]
    for k, v in kw.items():
        opts.append(f"{k}={v}")
    return f"read_csv_auto('{sqlpath(path)}', {', '.join(opts)})"


def pandas_chunks(path, usecols=None, dtype=None, parse_dates=None):
    """Fallback reader. Handles .csv and .csv.gz transparently."""
    return pd.read_csv(
        path, usecols=usecols, dtype=dtype, parse_dates=parse_dates,
        chunksize=CFG.pandas_chunk_rows, low_memory=False,
        compression="gzip" if str(path).endswith(".gz") else "infer",
    )

print("Engine:", "duckdb" if CON is not None else "pandas-chunked")

Engine: pandas-chunked


---
## 6. Analyte panel

The panel is the set of laboratory quantities that plausibly feed an ICU deterioration model. Each
canonical analyte is assigned a **turnaround class**, and that classification carries the internal
control for the whole study:

* `poc` — point of care and blood gas. Measured at or near the bedside. Should post almost immediately.
* `core` — routine automated chemistry and haematology. Central laboratory, moderate turnaround.
* `send` — specialised assays with longer processing.

If the latency measure is capturing something real, `poc` will be far faster than `send` **within the
same database and the same patients**. If it is not, that is a finding about the timestamp field
rather than about laboratory operations, and the study stops there.

Identifiers are resolved by matching each database's own dictionary rather than by hard-coded lists,
and the resolved mapping is printed for inspection.

In [7]:
# canonical analyte -> (turnaround class, MIMIC label regex, MIMIC fluid filter, eICU labname regex)
PANEL = {
    # ---- point of care / blood gas ----------------------------------------
    "ph":              ("poc",  r"^ph$",                              "blood", r"^ph$"),
    "pco2":            ("poc",  r"^pco2$",                            "blood", r"^paco2$"),
    "po2":             ("poc",  r"^po2$",                             "blood", r"^pao2$"),
    "base_excess":     ("poc",  r"^base excess$",                     "blood", r"^base excess$"),
    "lactate":         ("poc",  r"^lactate$",                         "blood", r"^lactate$"),
    "glucose":         ("core", r"^glucose$",                         "blood", r"^glucose$"),

    # ---- core chemistry ----------------------------------------------------
    "sodium":          ("core", r"^sodium$",                          "blood", r"^sodium$"),
    "potassium":       ("core", r"^potassium$",                       "blood", r"^potassium$"),
    "chloride":        ("core", r"^chloride$",                        "blood", r"^chloride$"),
    "bicarbonate":     ("core", r"^bicarbonate$",                     "blood", r"^bicarbonate$"),
    "anion_gap":       ("core", r"^anion gap$",                       "blood", r"^anion gap$"),
    "urea_nitrogen":   ("core", r"^urea nitrogen$",                   "blood", r"^bun$"),
    "creatinine":      ("core", r"^creatinine$",                      "blood", r"^creatinine$"),
    "calcium_total":   ("core", r"^calcium, total$",                  "blood", r"^calcium$"),
    "magnesium":       ("core", r"^magnesium$",                       "blood", r"^magnesium$"),
    "phosphate":       ("core", r"^phosphate$",                       "blood", r"^phosphate$"),
    "albumin":         ("core", r"^albumin$",                         "blood", r"^albumin$"),
    "bilirubin_total": ("core", r"^bilirubin, total$",                "blood", r"^total bilirubin$"),
    "alt":             ("core", r"^alanine aminotransferase \(alt\)$","blood", r"^alt \(sgpt\)$"),
    "ast":             ("core", r"^asparate aminotransferase \(ast\)$","blood", r"^ast \(sgot\)$"),

    # ---- haematology -------------------------------------------------------
    "wbc":             ("core", r"^white blood cells$",               "blood", r"^wbc x 1000$"),
    "hemoglobin":      ("core", r"^hemoglobin$",                      "blood", r"^hgb$"),
    "hematocrit":      ("core", r"^hematocrit$",                      "blood", r"^hct$"),
    "platelet":        ("core", r"^platelet count$",                  "blood", r"^platelets x 1000$"),

    # ---- coagulation -------------------------------------------------------
    "inr":             ("core", r"^inr\(pt\)$",                       "blood", r"^pt - inr$"),
    "pt":              ("core", r"^pt$",                              "blood", r"^pt$"),
    "ptt":             ("core", r"^ptt$",                             "blood", r"^ptt$"),

    # ---- longer turnaround assays -----------------------------------------
    "troponin_t":      ("send", r"^troponin t$",                      "blood", r"^troponin - t$"),
    "crp":             ("send", r"^c-reactive protein$",              "blood", r"^crp$"),
    "procalcitonin":   ("send", r"^procalcitonin$",                   "blood", r"^procalcitonin$"),
    "bnp":             ("send", r"^ntprobnp$",                        "blood", r"^bnp$"),
    "fibrinogen":      ("send", r"^fibrinogen, functional$",          "blood", r"^fibrinogen$"),
}

TURNAROUND_CLASS = {k: v[0] for k, v in PANEL.items()}
CLASS_ORDER = ["poc", "core", "send"]

print(f"{len(PANEL)} canonical analytes across {len(CLASS_ORDER)} turnaround classes")
print(pd.Series(TURNAROUND_CLASS).value_counts().reindex(CLASS_ORDER).to_string())

32 canonical analytes across 3 turnaround classes
poc      5
core    22
send     5


In [8]:
# ---- resolve MIMIC-IV itemids against d_labitems --------------------------
dlab = pd.read_csv(SRC["mimic.d_labitems"])
dlab.columns = [c.lower() for c in dlab.columns]
for c in ("label", "fluid", "category"):
    if c in dlab.columns:
        dlab[c] = dlab[c].astype(str).str.strip()

rows = []
for canon, (tclass, mrx, mfluid, _) in PANEL.items():
    m = dlab["label"].str.lower().str.match(mrx, na=False)
    if mfluid and "fluid" in dlab.columns:
        m &= dlab["fluid"].str.lower().str.contains(mfluid, na=False)
    hit = dlab.loc[m]
    for _, r in hit.iterrows():
        rows.append({
            "analyte": canon, "turnaround_class": tclass,
            "itemid": int(r["itemid"]), "label": r["label"],
            "fluid": r.get("fluid", ""), "category": r.get("category", ""),
        })

MIMIC_MAP = pd.DataFrame(rows).drop_duplicates("itemid").reset_index(drop=True)
MIMIC_ITEMIDS = MIMIC_MAP["itemid"].astype(int).tolist()

unmatched = sorted(set(PANEL) - set(MIMIC_MAP["analyte"]))
print(f"MIMIC-IV: {len(MIMIC_ITEMIDS)} itemids resolved for {MIMIC_MAP['analyte'].nunique()} of {len(PANEL)} analytes")
if unmatched:
    print("  not present in d_labitems:", ", ".join(unmatched))

display(MIMIC_MAP.sort_values(["turnaround_class", "analyte"]).reset_index(drop=True))
write_table(MIMIC_MAP, "t1a_analyte_map_mimic")

MIMIC-IV: 54 itemids resolved for 31 of 32 analytes
  not present in d_labitems: procalcitonin


,analyte,turnaround_class,itemid,label,fluid,category
0,albumin,core,50862,Albumin,Blood,Chemistry
1,albumin,core,53085,Albumin,Blood,Chemistry
2,alt,core,50861,Alanine Aminotransferase (ALT),Blood,Chemistry
3,anion_gap,core,50868,Anion Gap,Blood,Chemistry
4,anion_gap,core,52500,Anion Gap,Blood,Chemistry
5,ast,core,50878,Asparate Aminotransferase (AST),Blood,Chemistry
6,bicarbonate,core,50882,Bicarbonate,Blood,Chemistry
7,bilirubin_total,core,50885,"Bilirubin, Total",Blood,Chemistry
8,bilirubin_total,core,53089,"Bilirubin, Total",Blood,Chemistry
9,calcium_total,core,50893,"Calcium, Total",Blood,Chemistry


WindowsPath('C:/Research_Paper_2/result_availability_audit/tables/t1a_analyte_map_mimic.csv')

In [9]:
# ---- resolve eICU labnames by scanning the distinct set --------------------
with Stage("eicu-labname-scan"):
    if CON is not None:
        labnames = dsql(f"""
            SELECT labname, COUNT(*) AS n
            FROM {read_csv_expr(SRC['eicu.lab'])}
            GROUP BY labname
            ORDER BY n DESC
        """)
    else:
        acc = {}
        for ch in pandas_chunks(SRC["eicu.lab"], usecols=["labname"], dtype={"labname": "string"}):
            for k, v in ch["labname"].value_counts().items():
                acc[k] = acc.get(k, 0) + int(v)
        labnames = (pd.DataFrame({"labname": list(acc), "n": list(acc.values())})
                    .sort_values("n", ascending=False).reset_index(drop=True))

labnames["labname"] = labnames["labname"].astype(str).str.strip()

rows = []
for canon, (tclass, _, _, erx) in PANEL.items():
    hit = labnames.loc[labnames["labname"].str.lower().str.match(erx, na=False)]
    for _, r in hit.iterrows():
        rows.append({"analyte": canon, "turnaround_class": tclass,
                     "labname": r["labname"], "n_rows": int(r["n"])})

EICU_MAP = pd.DataFrame(rows).drop_duplicates("labname").reset_index(drop=True)
EICU_LABNAMES = EICU_MAP["labname"].tolist()

unmatched_e = sorted(set(PANEL) - set(EICU_MAP["analyte"]))
print(f"eICU: {len(EICU_LABNAMES)} labnames resolved for {EICU_MAP['analyte'].nunique()} of {len(PANEL)} analytes")
if unmatched_e:
    print("  not present in lab.labname:", ", ".join(unmatched_e))

display(EICU_MAP.sort_values(["turnaround_class", "analyte"]).reset_index(drop=True))
write_table(EICU_MAP, "t1b_analyte_map_eicu")

SHARED = sorted(set(MIMIC_MAP["analyte"]) & set(EICU_MAP["analyte"]))
print(f"\nAnalytes resolvable in BOTH databases ({len(SHARED)}): {', '.join(SHARED)}")

[eicu-labname-scan] start
[eicu-labname-scan] done in 48.9 s
eICU: 31 labnames resolved for 31 of 32 analytes
  not present in lab.labname: procalcitonin


,analyte,turnaround_class,labname,n_rows
0,albumin,core,albumin,506025
1,alt,core,ALT (SGPT),434605
2,anion_gap,core,anion gap,1024278
3,ast,core,AST (SGOT),437311
4,bicarbonate,core,bicarbonate,1199866
5,bilirubin_total,core,total bilirubin,427307
6,calcium_total,core,calcium,1226978
7,chloride,core,chloride,1283839
8,creatinine,core,creatinine,1277760
9,glucose,core,glucose,1319496



Analytes resolvable in BOTH databases (31): albumin, alt, anion_gap, ast, base_excess, bicarbonate, bilirubin_total, bnp, calcium_total, chloride, creatinine, crp, fibrinogen, glucose, hematocrit, hemoglobin, inr, lactate, magnesium, pco2, ph, phosphate, platelet, po2, potassium, pt, ptt, sodium, troponin_t, urea_nitrogen, wbc


---
## 7. Cohorts

One ICU stay per patient, adult, with a stay long enough to host a scoring window. Restricting to the
first stay removes within-patient clustering, which otherwise inflates the effective sample and
biases latency estimates towards frequently readmitted patients.

In [10]:
COHORT_MIMIC_P = DIR["cache"] / "cohort_mimic.parquet"

with Stage("cohort-mimic"):
    if cached(COHORT_MIMIC_P):
        cohort_m = pd.read_parquet(COHORT_MIMIC_P)
        print("loaded from cache")
    else:
        if CON is not None:
            rank_clause = "WHERE rn = 1" if CFG.first_stay_only else ""
            cohort_m = dsql(f"""
                WITH ie AS (
                    SELECT
                        TRY_CAST(subject_id AS BIGINT) AS subject_id,
                        TRY_CAST(hadm_id    AS BIGINT) AS hadm_id,
                        TRY_CAST(stay_id    AS BIGINT) AS stay_id,
                        first_careunit, last_careunit,
                        TRY_CAST(intime  AS TIMESTAMP) AS intime,
                        TRY_CAST(outtime AS TIMESTAMP) AS outtime,
                        TRY_CAST(los AS DOUBLE) AS los_days
                    FROM {read_csv_expr(SRC['mimic.icustays'])}
                ),
                pa AS (
                    SELECT
                        TRY_CAST(subject_id AS BIGINT) AS subject_id,
                        gender,
                        TRY_CAST(anchor_age  AS INTEGER) AS anchor_age,
                        TRY_CAST(anchor_year AS INTEGER) AS anchor_year,
                        anchor_year_group,
                        TRY_CAST(dod AS TIMESTAMP) AS dod
                    FROM {read_csv_expr(SRC['mimic.patients'])}
                ),
                ad AS (
                    SELECT
                        TRY_CAST(hadm_id AS BIGINT) AS hadm_id,
                        TRY_CAST(admittime AS TIMESTAMP) AS admittime,
                        TRY_CAST(dischtime AS TIMESTAMP) AS dischtime,
                        admission_type, admission_location, insurance, race,
                        TRY_CAST(hospital_expire_flag AS INTEGER) AS hospital_expire_flag
                    FROM {read_csv_expr(SRC['mimic.admissions'])}
                ),
                j AS (
                    SELECT
                        ie.*, pa.gender, pa.anchor_year_group, pa.dod,
                        ad.admittime, ad.dischtime, ad.admission_type,
                        ad.admission_location, ad.insurance, ad.race,
                        ad.hospital_expire_flag,
                        pa.anchor_age + (EXTRACT(YEAR FROM ie.intime) - pa.anchor_year) AS age_years,
                        DATE_DIFF('minute', ie.intime, ie.outtime) / 60.0 AS icu_hours,
                        ROW_NUMBER() OVER (PARTITION BY ie.subject_id ORDER BY ie.intime) AS rn
                    FROM ie
                    JOIN pa USING (subject_id)
                    LEFT JOIN ad USING (hadm_id)
                )
                SELECT * FROM j
                {rank_clause}
            """)
        else:
            ie = pd.read_csv(SRC["mimic.icustays"], parse_dates=["intime", "outtime"])
            pts = pd.read_csv(SRC["mimic.patients"], parse_dates=["dod"])
            adm = pd.read_csv(SRC["mimic.admissions"],
                              usecols=["hadm_id", "admittime", "dischtime", "admission_type",
                                       "admission_location", "insurance", "race", "hospital_expire_flag"],
                              parse_dates=["admittime", "dischtime"])
            cohort_m = ie.merge(pts, on="subject_id", how="inner").merge(adm, on="hadm_id", how="left")
            cohort_m["age_years"] = cohort_m["anchor_age"] + (cohort_m["intime"].dt.year - cohort_m["anchor_year"])
            cohort_m["icu_hours"] = (cohort_m["outtime"] - cohort_m["intime"]).dt.total_seconds() / 3600
            cohort_m["los_days"] = cohort_m["los"]
            cohort_m["rn"] = cohort_m.sort_values("intime").groupby("subject_id").cumcount() + 1
            if CFG.first_stay_only:
                cohort_m = cohort_m[cohort_m["rn"] == 1]

        n0 = len(cohort_m)
        steps = [("all first ICU stays" if CFG.first_stay_only else "all ICU stays", n0)]

        cohort_m = cohort_m[cohort_m["age_years"].between(CFG.min_age_years, CFG.max_age_years)]
        steps.append((f"age {CFG.min_age_years}-{CFG.max_age_years}", len(cohort_m)))

        cohort_m = cohort_m[cohort_m["icu_hours"].between(CFG.min_icu_hours, CFG.max_icu_hours)]
        steps.append((f"ICU stay {CFG.min_icu_hours:.0f}-{CFG.max_icu_hours:.0f} h", len(cohort_m)))

        cohort_m = cohort_m[cohort_m["intime"].notna() & cohort_m["outtime"].notna()]
        steps.append(("valid ICU timestamps", len(cohort_m)))

        cohort_m = cohort_m.drop(columns=["rn"], errors="ignore").reset_index(drop=True)
        cohort_m["db"] = "MIMIC-IV"
        write_parquet(cohort_m, "cohort_mimic")

        flow_m = pd.DataFrame(steps, columns=["step", "n_stays"])
        flow_m["excluded"] = flow_m["n_stays"].shift(1).fillna(flow_m["n_stays"]).astype(int) - flow_m["n_stays"]
        write_table(flow_m, "t2a_cohort_flow_mimic")
        display(flow_m)

print(f"MIMIC-IV analytic cohort: {len(cohort_m):,} ICU stays, {cohort_m['subject_id'].nunique():,} patients")
MIMIC_STAY_KEYS = cohort_m[["subject_id", "hadm_id", "stay_id", "intime", "outtime"]].copy()

[cohort-mimic] start
loaded from cache
[cohort-mimic] done in 0.1 s
MIMIC-IV analytic cohort: 48,736 ICU stays, 48,736 patients


In [11]:
COHORT_EICU_P = DIR["cache"] / "cohort_eicu.parquet"

with Stage("cohort-eicu"):
    if cached(COHORT_EICU_P):
        cohort_e = pd.read_parquet(COHORT_EICU_P)
        print("loaded from cache")
    else:
        if CON is not None:
            cohort_e = dsql(f"""
                WITH p AS (
                    SELECT
                        TRY_CAST(patientunitstayid AS BIGINT) AS patientunitstayid,
                        TRY_CAST(patienthealthsystemstayid AS BIGINT) AS patienthealthsystemstayid,
                        uniquepid, gender, age AS age_raw, ethnicity,
                        TRY_CAST(hospitalid AS BIGINT) AS hospitalid,
                        unittype, unitadmitsource, unitstaytype,
                        TRY_CAST(unitvisitnumber AS INTEGER) AS unitvisitnumber,
                        TRY_CAST(unitdischargeoffset AS DOUBLE) AS unitdischargeoffset,
                        unitdischargestatus, hospitaldischargestatus,
                        TRY_CAST(hospitaldischargeyear AS INTEGER) AS hospitaldischargeyear,
                        apacheadmissiondx,
                        CASE
                            WHEN TRIM(age) IN ('> 89', '>89') THEN 90
                            ELSE TRY_CAST(TRIM(age) AS INTEGER)
                        END AS age_years
                    FROM {read_csv_expr(SRC['eicu.patient'])}
                ),
                h AS (
                    SELECT
                        TRY_CAST(hospitalid AS BIGINT) AS hospitalid,
                        numbedscategory, teachingstatus, region
                    FROM {read_csv_expr(SRC['eicu.hospital'])}
                ),
                j AS (
                    SELECT p.*, h.numbedscategory, h.teachingstatus, h.region,
                           p.unitdischargeoffset / 60.0 AS icu_hours,
                           ROW_NUMBER() OVER (
                               PARTITION BY p.uniquepid
                               ORDER BY p.patienthealthsystemstayid, p.unitvisitnumber, p.patientunitstayid
                           ) AS rn
                    FROM p LEFT JOIN h USING (hospitalid)
                )
                SELECT * FROM j WHERE unitvisitnumber = 1
            """)
        else:
            p = pd.read_csv(SRC["eicu.patient"], low_memory=False)
            h = pd.read_csv(SRC["eicu.hospital"])
            p["age_years"] = pd.to_numeric(
                p["age"].astype(str).str.strip().replace({"> 89": "90", ">89": "90"}), errors="coerce")
            cohort_e = p.merge(h, on="hospitalid", how="left")
            cohort_e["icu_hours"] = cohort_e["unitdischargeoffset"] / 60.0
            cohort_e = cohort_e[cohort_e["unitvisitnumber"] == 1]
            cohort_e["rn"] = (cohort_e.sort_values(["patienthealthsystemstayid", "patientunitstayid"])
                              .groupby("uniquepid").cumcount() + 1)

        n0 = len(cohort_e)
        steps = [("first unit stay of hospital admission", n0)]

        if CFG.first_stay_only:
            cohort_e = cohort_e[cohort_e["rn"] == 1]
            steps.append(("first admission per patient", len(cohort_e)))

        cohort_e = cohort_e[cohort_e["age_years"].between(CFG.min_age_years, CFG.max_age_years)]
        steps.append((f"age {CFG.min_age_years}-{CFG.max_age_years}", len(cohort_e)))

        cohort_e = cohort_e[cohort_e["icu_hours"].between(CFG.min_icu_hours, CFG.max_icu_hours)]
        steps.append((f"ICU stay {CFG.min_icu_hours:.0f}-{CFG.max_icu_hours:.0f} h", len(cohort_e)))

        cohort_e = cohort_e.drop(columns=["rn"], errors="ignore").reset_index(drop=True)
        cohort_e["db"] = "eICU-CRD"
        write_parquet(cohort_e, "cohort_eicu")

        flow_e = pd.DataFrame(steps, columns=["step", "n_stays"])
        flow_e["excluded"] = flow_e["n_stays"].shift(1).fillna(flow_e["n_stays"]).astype(int) - flow_e["n_stays"]
        write_table(flow_e, "t2b_cohort_flow_eicu")
        display(flow_e)

print(f"eICU analytic cohort: {len(cohort_e):,} ICU stays, "
      f"{cohort_e['uniquepid'].nunique():,} patients, {cohort_e['hospitalid'].nunique()} hospitals")

[cohort-eicu] start
loaded from cache
[cohort-eicu] done in 0.3 s
eICU analytic cohort: 120,354 ICU stays, 120,354 patients, 208 hospitals


In [12]:
# ---- side-by-side cohort description --------------------------------------
def describe_cohort(df, db, id_col, extra=None):
    n = len(df)
    d = {
        "database": db,
        "ICU stays": f"{n:,}",
        "Patients": f"{df[id_col].nunique():,}",
        "Age, median (IQR)": f"{df['age_years'].median():.0f} ({df['age_years'].quantile(.25):.0f}-{df['age_years'].quantile(.75):.0f})",
        "Female, n (%)": f"{(df['gender'].astype(str).str.upper().str.startswith('F')).sum():,} "
                         f"({100*(df['gender'].astype(str).str.upper().str.startswith('F')).mean():.1f}%)",
        "ICU hours, median (IQR)": f"{df['icu_hours'].median():.1f} ({df['icu_hours'].quantile(.25):.1f}-{df['icu_hours'].quantile(.75):.1f})",
    }
    if extra:
        d.update(extra)
    return d

mort_m = cohort_m["hospital_expire_flag"].fillna(0).astype(int)
rows = [
    describe_cohort(cohort_m, "MIMIC-IV", "subject_id", {
        "Sites": "1",
        "In-hospital mortality, n (%)": f"{mort_m.sum():,} ({100*mort_m.mean():.1f}%)",
    }),
    describe_cohort(cohort_e, "eICU-CRD", "uniquepid", {
        "Sites": f"{cohort_e['hospitalid'].nunique()}",
        "In-hospital mortality, n (%)": (
            lambda s: f"{(s=='Expired').sum():,} ({100*(s=='Expired').mean():.1f}%)"
        )(cohort_e["hospitaldischargestatus"].astype(str)),
    }),
]
tbl1 = pd.DataFrame(rows).set_index("database").T
display(tbl1)
write_table(tbl1, "t3_cohort_characteristics", index=True)

database,MIMIC-IV,eICU-CRD
ICU stays,"48,736","120,354"
Patients,"48,736","120,354"
"Age, median (IQR)",66 (54-78),65 (53-77)
"Female, n (%)","21,506 (44.1%)","55,101 (45.8%)"
"ICU hours, median (IQR)",46.2 (27.2-86.6),43.4 (24.6-76.2)
Sites,1,208
"In-hospital mortality, n (%)","4,694 (9.6%)","10,142 (8.4%)"


WindowsPath('C:/Research_Paper_2/result_availability_audit/tables/t3_cohort_characteristics.csv')

---
## 8. Laboratory extraction with both clocks

This is the single expensive pass. The cohort keys and the analyte identifiers are pushed into the
scan so only the rows that matter are ever materialised. Everything downstream reads the Parquet
output rather than the CSV.

**MIMIC-IV.** `charttime` is the observation clock, `storetime` the availability clock. The
`priority` field separates urgent from routine ordering, which is the operational variable most
likely to modify turnaround.

**eICU-CRD.** `labresultoffset` is the observation clock. `labresultrevisedoffset` is documented as
the time a *revised* value was entered, so it stands in for availability rather than measuring it
directly. Section 10 quantifies how often the two coincide, so the strength of that proxy is
reported rather than assumed.

In [13]:
LABS_MIMIC_P = DIR["cache"] / "labs_mimic.parquet"

with Stage("labs-mimic"):
    if cached(LABS_MIMIC_P):
        labs_m = pd.read_parquet(LABS_MIMIC_P)
        print("loaded from cache")
    else:
        itemid_list = ",".join(str(i) for i in MIMIC_ITEMIDS)
        if CON is not None:
            CON.register("coh_m", MIMIC_STAY_KEYS)
            CON.register("map_m", MIMIC_MAP[["itemid", "analyte", "turnaround_class"]])
            CON.execute(f"""
                CREATE OR REPLACE TABLE labs_m AS
                WITH le AS (
                    SELECT
                        TRY_CAST(subject_id AS BIGINT)  AS subject_id,
                        TRY_CAST(hadm_id    AS BIGINT)  AS hadm_id,
                        TRY_CAST(itemid     AS INTEGER) AS itemid,
                        TRY_CAST(charttime  AS TIMESTAMP) AS charttime,
                        TRY_CAST(storetime  AS TIMESTAMP) AS storetime,
                        TRY_CAST(valuenum   AS DOUBLE)  AS valuenum,
                        CAST(valueuom AS VARCHAR)  AS valueuom,
                        CAST(flag     AS VARCHAR)  AS flag,
                        CAST(priority AS VARCHAR)  AS priority
                    FROM {read_csv_expr(SRC['mimic.labevents'])}
                    WHERE TRY_CAST(itemid AS INTEGER) IN ({itemid_list})
                )
                SELECT
                    c.stay_id, le.subject_id, le.hadm_id,
                    m.analyte, m.turnaround_class, le.itemid,
                    le.charttime, le.storetime, le.valuenum, le.valueuom, le.flag,
                    COALESCE(NULLIF(TRIM(le.priority), ''), 'UNSPECIFIED') AS priority,
                    DATE_DIFF('minute', le.charttime, le.storetime) AS latency_min,
                    DATE_DIFF('minute', c.intime, le.charttime) / 60.0 AS hours_from_icu_admit,
                    (le.charttime >= c.intime AND le.charttime <= c.outtime) AS in_icu,
                    EXTRACT(HOUR FROM le.charttime)  AS draw_hour,
                    EXTRACT(DOW  FROM le.charttime)  AS draw_dow,
                    EXTRACT(YEAR FROM le.charttime)  AS draw_year
                FROM le
                JOIN coh_m c
                  ON le.subject_id = c.subject_id
                 AND le.charttime >= c.intime - INTERVAL {int(CFG.pre_icu_hours)} HOUR
                 AND le.charttime <= c.outtime
                JOIN map_m m ON le.itemid = m.itemid
                WHERE le.charttime IS NOT NULL
            """)
            labs_m = CON.execute("SELECT * FROM labs_m").df()
        else:
            keep = set(MIMIC_ITEMIDS)
            coh = MIMIC_STAY_KEYS.set_index("subject_id")
            parts = []
            for i, ch in enumerate(pandas_chunks(
                    SRC["mimic.labevents"],
                    usecols=["subject_id", "hadm_id", "itemid", "charttime", "storetime",
                             "valuenum", "valueuom", "flag", "priority"],
                    parse_dates=["charttime", "storetime"])):
                ch = ch[ch["itemid"].isin(keep)]
                if ch.empty:
                    continue
                ch = ch.join(coh, on="subject_id", how="inner", rsuffix="_c")
                lo = ch["intime"] - pd.Timedelta(hours=CFG.pre_icu_hours)
                ch = ch[(ch["charttime"] >= lo) & (ch["charttime"] <= ch["outtime"])]
                if not ch.empty:
                    parts.append(ch)
                if i % 10 == 0:
                    print(f"  chunk {i}, kept so far {sum(len(p) for p in parts):,}")
            labs_m = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()
            labs_m = labs_m.merge(MIMIC_MAP[["itemid", "analyte", "turnaround_class"]], on="itemid", how="inner")
            labs_m["priority"] = labs_m["priority"].fillna("").str.strip().replace("", "UNSPECIFIED")
            labs_m["latency_min"] = (labs_m["storetime"] - labs_m["charttime"]).dt.total_seconds() / 60
            labs_m["hours_from_icu_admit"] = (labs_m["charttime"] - labs_m["intime"]).dt.total_seconds() / 3600
            labs_m["in_icu"] = (labs_m["charttime"] >= labs_m["intime"]) & (labs_m["charttime"] <= labs_m["outtime"])
            labs_m["draw_hour"] = labs_m["charttime"].dt.hour
            labs_m["draw_dow"] = labs_m["charttime"].dt.dayofweek
            labs_m["draw_year"] = labs_m["charttime"].dt.year
            labs_m = labs_m.drop(columns=["intime", "outtime", "hadm_id_c"], errors="ignore")

        labs_m["latency_min"] = pd.to_numeric(labs_m["latency_min"], errors="coerce")
        labs_m["db"] = "MIMIC-IV"
        write_parquet(labs_m, "labs_mimic")

print(f"MIMIC-IV laboratory rows: {len(labs_m):,} "
      f"across {labs_m['stay_id'].nunique():,} stays and {labs_m['analyte'].nunique()} analytes")
print(f"Memory: {labs_m.memory_usage(deep=True).sum()/(1<<30):.2f} GB")

[labs-mimic] start
loaded from cache
[labs-mimic] done in 3.6 s
MIMIC-IV laboratory rows: 6,579,894 across 48,602 stays and 31 analytes
Memory: 2.71 GB


In [14]:
LABS_EICU_P = DIR["cache"] / "labs_eicu.parquet"

with Stage("labs-eicu"):
    if cached(LABS_EICU_P):
        labs_e = pd.read_parquet(LABS_EICU_P)
        print("loaded from cache")
    else:
        name_list = ",".join("'" + n.replace("'", "''") + "'" for n in EICU_LABNAMES)
        coh_keys_e = cohort_e[["patientunitstayid", "hospitalid", "unitdischargeoffset"]].copy()
        if CON is not None:
            CON.register("coh_e", coh_keys_e)
            CON.register("map_e", EICU_MAP[["labname", "analyte", "turnaround_class"]])
            CON.execute(f"""
                CREATE OR REPLACE TABLE labs_e AS
                WITH lb AS (
                    SELECT
                        TRY_CAST(patientunitstayid       AS BIGINT) AS patientunitstayid,
                        TRIM(CAST(labname AS VARCHAR))              AS labname,
                        TRY_CAST(labresultoffset         AS DOUBLE) AS obs_offset_min,
                        TRY_CAST(labresultrevisedoffset  AS DOUBLE) AS entry_offset_min,
                        TRY_CAST(labresult               AS DOUBLE) AS labresult,
                        CAST(labmeasurenamesystem AS VARCHAR)       AS uom
                    FROM {read_csv_expr(SRC['eicu.lab'])}
                    WHERE TRIM(CAST(labname AS VARCHAR)) IN ({name_list})
                )
                SELECT
                    lb.patientunitstayid, c.hospitalid,
                    m.analyte, m.turnaround_class, lb.labname,
                    lb.obs_offset_min, lb.entry_offset_min, lb.labresult, lb.uom,
                    (lb.entry_offset_min - lb.obs_offset_min) AS latency_min,
                    lb.obs_offset_min / 60.0 AS hours_from_icu_admit,
                    (lb.obs_offset_min >= 0 AND lb.obs_offset_min <= c.unitdischargeoffset) AS in_icu
                FROM lb
                JOIN coh_e c USING (patientunitstayid)
                JOIN map_e m ON lb.labname = m.labname
                WHERE lb.obs_offset_min IS NOT NULL
                  AND lb.obs_offset_min >= -{CFG.pre_icu_hours * 60}
                  AND lb.obs_offset_min <= c.unitdischargeoffset
            """)
            labs_e = CON.execute("SELECT * FROM labs_e").df()
        else:
            keep = set(EICU_LABNAMES)
            coh = coh_keys_e.set_index("patientunitstayid")
            parts = []
            for i, ch in enumerate(pandas_chunks(
                    SRC["eicu.lab"],
                    usecols=["patientunitstayid", "labname", "labresultoffset",
                             "labresultrevisedoffset", "labresult", "labmeasurenamesystem"])):
                ch["labname"] = ch["labname"].astype(str).str.strip()
                ch = ch[ch["labname"].isin(keep)]
                if ch.empty:
                    continue
                ch = ch.join(coh, on="patientunitstayid", how="inner")
                ch = ch[(ch["labresultoffset"] >= -CFG.pre_icu_hours * 60) &
                        (ch["labresultoffset"] <= ch["unitdischargeoffset"])]
                if not ch.empty:
                    parts.append(ch)
                if i % 5 == 0:
                    print(f"  chunk {i}, kept so far {sum(len(p) for p in parts):,}")
            labs_e = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()
            labs_e = labs_e.rename(columns={"labresultoffset": "obs_offset_min",
                                            "labresultrevisedoffset": "entry_offset_min",
                                            "labmeasurenamesystem": "uom"})
            labs_e = labs_e.merge(EICU_MAP[["labname", "analyte", "turnaround_class"]], on="labname", how="inner")
            labs_e["latency_min"] = labs_e["entry_offset_min"] - labs_e["obs_offset_min"]
            labs_e["hours_from_icu_admit"] = labs_e["obs_offset_min"] / 60.0
            labs_e["in_icu"] = (labs_e["obs_offset_min"] >= 0) & (labs_e["obs_offset_min"] <= labs_e["unitdischargeoffset"])
            labs_e = labs_e.drop(columns=["unitdischargeoffset"], errors="ignore")

        labs_e["latency_min"] = pd.to_numeric(labs_e["latency_min"], errors="coerce")
        labs_e["db"] = "eICU-CRD"
        write_parquet(labs_e, "labs_eicu")

print(f"eICU laboratory rows: {len(labs_e):,} across "
      f"{labs_e['patientunitstayid'].nunique():,} stays, {labs_e['hospitalid'].nunique()} hospitals, "
      f"{labs_e['analyte'].nunique()} analytes")

[labs-eicu] start
loaded from cache
[labs-eicu] done in 6.2 s
eICU laboratory rows: 9,367,414 across 118,212 stays, 206 hospitals, 31 analytes


In [15]:
# ---- microbiology: the longest turnaround in the record --------------------
MICRO_P = DIR["cache"] / "micro_mimic.parquet"

with Stage("micro-mimic"):
    if cached(MICRO_P):
        micro = pd.read_parquet(MICRO_P)
        print("loaded from cache")
    else:
        if CON is not None:
            CON.register("coh_m2", MIMIC_STAY_KEYS)
            micro = dsql(f"""
                WITH mb AS (
                    SELECT
                        TRY_CAST(subject_id AS BIGINT) AS subject_id,
                        TRY_CAST(hadm_id    AS BIGINT) AS hadm_id,
                        COALESCE(TRY_CAST(charttime AS TIMESTAMP),
                                 TRY_CAST(chartdate AS TIMESTAMP)) AS charttime,
                        COALESCE(TRY_CAST(storetime AS TIMESTAMP),
                                 TRY_CAST(storedate AS TIMESTAMP)) AS storetime,
                        CAST(spec_type_desc AS VARCHAR) AS specimen,
                        CAST(test_name     AS VARCHAR) AS test_name,
                        CAST(org_name      AS VARCHAR) AS org_name,
                        CAST(interpretation AS VARCHAR) AS interpretation
                    FROM {read_csv_expr(SRC['mimic.microbiologyevents'])}
                )
                SELECT
                    c.stay_id, mb.subject_id, mb.hadm_id, mb.charttime, mb.storetime,
                    mb.specimen, mb.test_name, mb.org_name, mb.interpretation,
                    (mb.org_name IS NOT NULL AND TRIM(mb.org_name) <> '') AS organism_isolated,
                    DATE_DIFF('minute', mb.charttime, mb.storetime) AS latency_min,
                    DATE_DIFF('minute', c.intime, mb.charttime) / 60.0 AS hours_from_icu_admit
                FROM mb
                JOIN coh_m2 c
                  ON mb.subject_id = c.subject_id
                 AND mb.charttime >= c.intime - INTERVAL {int(CFG.pre_icu_hours)} HOUR
                 AND mb.charttime <= c.outtime
                WHERE mb.charttime IS NOT NULL AND mb.storetime IS NOT NULL
            """)
        else:
            mb = pd.read_csv(
                SRC["mimic.microbiologyevents"],
                usecols=["subject_id", "hadm_id", "chartdate", "charttime", "storedate",
                         "storetime", "spec_type_desc", "test_name", "org_name", "interpretation"],
                parse_dates=["chartdate", "charttime", "storedate", "storetime"], low_memory=False)
            mb["charttime"] = mb["charttime"].fillna(mb["chartdate"])
            mb["storetime"] = mb["storetime"].fillna(mb["storedate"])
            coh = MIMIC_STAY_KEYS.set_index("subject_id")
            mb = mb.join(coh, on="subject_id", how="inner", rsuffix="_c")
            lo = mb["intime"] - pd.Timedelta(hours=CFG.pre_icu_hours)
            micro = mb[(mb["charttime"] >= lo) & (mb["charttime"] <= mb["outtime"])].copy()
            micro["organism_isolated"] = micro["org_name"].notna() & (micro["org_name"].astype(str).str.strip() != "")
            micro["latency_min"] = (micro["storetime"] - micro["charttime"]).dt.total_seconds() / 60
            micro["hours_from_icu_admit"] = (micro["charttime"] - micro["intime"]).dt.total_seconds() / 3600
            micro = micro.rename(columns={"spec_type_desc": "specimen"})
            micro = micro.drop(columns=["intime", "outtime", "chartdate", "storedate", "hadm_id_c"], errors="ignore")

        micro["latency_min"] = pd.to_numeric(micro["latency_min"], errors="coerce")
        write_parquet(micro, "micro_mimic")

print(f"Microbiology rows: {len(micro):,} across {micro['stay_id'].nunique():,} stays")

[micro-mimic] start
loaded from cache
[micro-mimic] done in 0.3 s
Microbiology rows: 319,130 across 39,722 stays


---
## 9. Latency validation

A derived interval is only usable once its failure modes are quantified. This section reports them
before any analysis depends on the variable, and applies the exclusions explicitly so the analytic
denominator is traceable.

Four failure modes are checked:

* **Missing** — no availability timestamp at all.
* **Negative** — availability recorded before observation. A timestamp inconsistency, not a delay.
* **Exactly zero** — the two clocks coincide. Legitimate for bedside measurement, but a high rate in
  central laboratory analytes would indicate the field records something other than result release.
* **Implausibly long** — beyond the configured ceiling. Retained in the reported counts, excluded
  from the summary statistics, so the exclusion is visible rather than silent.

In [16]:
def latency_quality(df, label, max_min):
    n = len(df)
    lat = df["latency_min"]
    miss = lat.isna()
    neg  = lat < CFG.latency_min_minutes
    zero = lat == 0
    long = lat > max_min
    ok   = (~miss) & (~neg) & (lat <= max_min)
    return {
        "source": label,
        "n_results": n,
        "missing_n": int(miss.sum()),          "missing_pct": round(100*miss.mean(), 3),
        "negative_n": int(neg.sum()),          "negative_pct": round(100*neg.mean(), 3),
        "exact_zero_n": int(zero.sum()),       "exact_zero_pct": round(100*zero.mean(), 3),
        f"gt_{int(max_min/60)}h_n": int(long.sum()),
        f"gt_{int(max_min/60)}h_pct": round(100*long.mean(), 3),
        "analytic_n": int(ok.sum()),           "analytic_pct": round(100*ok.mean(), 3),
    }

qa = pd.DataFrame([
    latency_quality(labs_m, "MIMIC-IV laboratory", CFG.latency_max_minutes),
    latency_quality(labs_e, "eICU-CRD laboratory", CFG.latency_max_minutes),
    latency_quality(micro,  "MIMIC-IV microbiology", CFG.micro_latency_max_minutes),
])
display(qa.T)
write_table(qa, "t4_latency_data_quality")

,0,1,2
source,MIMIC-IV laboratory,eICU-CRD laboratory,MIMIC-IV microbiology
n_results,6579894,9367414,319130
missing_n,1,0,1167
missing_pct,0.000,0.000,0.366
negative_n,193,568892,1
negative_pct,0.003,6.073,0.000
exact_zero_n,1852,1893470,0
exact_zero_pct,0.028,20.213,0.000
gt_24h_n,"1,353.000","21,580.000",NaN
gt_24h_pct,0.021,0.230,NaN


WindowsPath('C:/Research_Paper_2/result_availability_audit/tables/t4_latency_data_quality.csv')

In [17]:
# ---- eICU proxy interrogation ---------------------------------------------
# The second eICU offset records entry of a revised value. If it coincides with the observation
# offset for most results, it is a weak proxy for availability and the paper must say so.
e = labs_e["latency_min"]
proxy = pd.DataFrame([{
    "results with entry offset present": f"{e.notna().sum():,} ({100*e.notna().mean():.1f}%)",
    "entry offset identical to observation offset": f"{(e == 0).sum():,} ({100*(e == 0).mean():.1f}%)",
    "entry offset earlier than observation offset": f"{(e < 0).sum():,} ({100*(e < 0).mean():.1f}%)",
    "entry offset later, 1-60 min": f"{e.between(1, 60).sum():,} ({100*e.between(1, 60).mean():.1f}%)",
    "entry offset later, 61-360 min": f"{e.between(61, 360).sum():,} ({100*e.between(61, 360).mean():.1f}%)",
    "entry offset later, >360 min": f"{(e > 360).sum():,} ({100*(e > 360).mean():.1f}%)",
}]).T.rename(columns={0: "eICU-CRD"})
display(proxy)
write_table(proxy, "t5_eicu_proxy_interrogation", index=True)

nonzero_share = float((e > 0).mean())
print(f"\nInterpretation: a non-zero interval is observed for {100*nonzero_share:.1f}% of eICU results.")
if nonzero_share < 0.25:
    print("  The revised-result offset behaves as a revision flag rather than a release time.")
    print("  eICU should be reported as a cross-site case-mix and ordering-intensity comparison,")
    print("  with latency quantification confined to MIMIC-IV.")
else:
    print("  The revised-result offset carries usable timing information and can support a")
    print("  secondary, explicitly labelled latency estimate.")

,eICU-CRD
results with entry offset present,"9,367,414 (100.0%)"
entry offset identical to observation offset,"1,893,470 (20.2%)"
entry offset earlier than observation offset,"568,892 (6.1%)"
"entry offset later, 1-60 min","5,053,981 (54.0%)"
"entry offset later, 61-360 min","1,780,178 (19.0%)"
"entry offset later, >360 min","70,893 (0.8%)"



Interpretation: a non-zero interval is observed for 73.7% of eICU results.
  The revised-result offset carries usable timing information and can support a
  secondary, explicitly labelled latency estimate.


In [18]:
# ---- apply exclusions, retain the analytic sets ---------------------------
def analytic_subset(df, max_min):
    lat = df["latency_min"]
    keep = lat.notna() & (lat >= CFG.latency_min_minutes) & (lat <= max_min)
    return df.loc[keep].copy(), int((~keep).sum())

labs_m_a, drop_m = analytic_subset(labs_m, CFG.latency_max_minutes)
labs_e_a, drop_e = analytic_subset(labs_e, CFG.latency_max_minutes)
micro_a,  drop_c = analytic_subset(micro,  CFG.micro_latency_max_minutes)

labs_m_a["latency_h"] = labs_m_a["latency_min"] / 60.0
labs_e_a["latency_h"] = labs_e_a["latency_min"] / 60.0
micro_a["latency_h"]  = micro_a["latency_min"] / 60.0

print(f"MIMIC-IV labs      : {len(labs_m_a):,} analytic  ({drop_m:,} excluded)")
print(f"eICU labs          : {len(labs_e_a):,} analytic  ({drop_e:,} excluded)")
print(f"MIMIC-IV micro     : {len(micro_a):,} analytic  ({drop_c:,} excluded)")

gc.collect()

MIMIC-IV labs      : 6,578,347 analytic  (1,547 excluded)
eICU labs          : 8,776,942 analytic  (590,472 excluded)
MIMIC-IV micro     : 309,328 analytic  (9,802 excluded)


34

In [19]:
# ---- correct turnaround class from the database's own category field -------
# Blood gas panel versions of glucose, hemoglobin and hematocrit resolve to itemids
# in the Blood Gas category and post at bedside speed. Classify from `category`
# rather than from the analyte name.
cat = dlab[["itemid", "category"]].copy()
cat["itemid"] = cat["itemid"].astype(int)
cat_map = cat.set_index("itemid")["category"]

for _df in (labs_m, labs_m_a):
    _df["itemid"] = _df["itemid"].astype(int)
    _df.drop(columns=["category"], errors="ignore", inplace=True)
    _df["category"] = _df["itemid"].map(cat_map)
    _df.loc[_df["category"].eq("Blood Gas"), "turnaround_class"] = "poc"

moved = int(labs_m_a["category"].eq("Blood Gas").sum())
print(f"Reclassified {moved:,} blood gas results to point of care")
display(labs_m_a.groupby("turnaround_class")["latency_min"].agg(n="size", median="median"))

Reclassified 1,418,313 blood gas results to point of care


,n,median
turnaround_class,,
core,5070782,60.000
poc,1418314,3.000
send,89251,72.000


---
## 10. Latency descriptives

Latency is right-skewed, so it is summarised by quantiles throughout. The upper quantiles are the
operationally relevant ones: a model firing on an hourly schedule is defeated by the tail, not by the
median.

In [20]:
QUANTS = [0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]

QCOLS = [f"p{int(round(q*100))}" for q in QUANTS]

def latency_summary(df, by, value="latency_min"):
    keys = by if isinstance(by, list) else [by]
    if len(df) == 0:
        return pd.DataFrame(columns=keys + ["n", "mean"] + QCOLS)
    g = df.groupby(by, dropna=False)[value]
    out = g.agg(n="size", mean="mean").reset_index()
    q = g.quantile(QUANTS).unstack()
    q.columns = [f"p{int(round(c*100))}" for c in q.columns]
    out = out.merge(q.reset_index(), on=keys)
    out["n"] = out["n"].astype(int)
    num = [c for c in out.columns if c not in keys and c != "n"]
    out[num] = out[num].round(1)
    return out


def fmt_iqr(df, unit="min"):
    d = df.copy()
    d[f"median ({unit})"] = d["p50"]
    d[f"IQR ({unit})"] = d["p25"].map(lambda v: f"{v:,.0f}") + "-" + d["p75"].map(lambda v: f"{v:,.0f}")
    d[f"p90 ({unit})"] = d["p90"]
    d[f"p99 ({unit})"] = d["p99"]
    return d

In [21]:
# ---- headline: latency by analyte and turnaround class --------------------
lat_analyte_m = latency_summary(labs_m_a, ["turnaround_class", "analyte"])
lat_analyte_m["db"] = "MIMIC-IV"
lat_analyte_e = latency_summary(labs_e_a, ["turnaround_class", "analyte"])
lat_analyte_e["db"] = "eICU-CRD"

lat_analyte = pd.concat([lat_analyte_m, lat_analyte_e], ignore_index=True)
lat_analyte["turnaround_class"] = pd.Categorical(lat_analyte["turnaround_class"],
                                                 categories=CLASS_ORDER, ordered=True)
lat_analyte = lat_analyte.sort_values(["db", "turnaround_class", "p50"]).reset_index(drop=True)

display(fmt_iqr(lat_analyte)[["db", "turnaround_class", "analyte", "n",
                              "median (min)", "IQR (min)", "p90 (min)", "p99 (min)"]])
write_table(lat_analyte, "t6_latency_by_analyte")

,db,turnaround_class,analyte,n,median (min),IQR (min),p90 (min),p99 (min)
0,MIMIC-IV,poc,base_excess,262499,3.000,2-4,5.000,18.000
1,MIMIC-IV,poc,glucose,112676,3.000,2-4,6.000,18.000
2,MIMIC-IV,poc,hemoglobin,57778,3.000,2-5,7.000,23.000
3,MIMIC-IV,poc,lactate,172679,3.000,2-4,6.000,26.000
4,MIMIC-IV,poc,pco2,262456,3.000,2-4,5.000,18.000
5,MIMIC-IV,poc,ph,287651,3.000,2-4,5.000,17.000
6,MIMIC-IV,poc,po2,262575,3.000,2-4,5.000,18.000
7,MIMIC-IV,core,hematocrit,303188,34.000,24-49,69.000,141.000
8,MIMIC-IV,core,hemoglobin,269290,35.000,24-49,70.000,149.000
9,MIMIC-IV,core,wbc,265225,35.000,25-50,73.000,157.000


WindowsPath('C:/Research_Paper_2/result_availability_audit/tables/t6_latency_by_analyte.csv')

In [22]:
# ---- the internal control: does turnaround class behave as expected? -------
ctrl = latency_summary(labs_m_a, "turnaround_class")
ctrl["turnaround_class"] = pd.Categorical(ctrl["turnaround_class"], categories=CLASS_ORDER, ordered=True)
ctrl = ctrl.sort_values("turnaround_class")
display(fmt_iqr(ctrl)[["turnaround_class", "n", "median (min)", "IQR (min)", "p90 (min)", "p99 (min)"]])

poc  = labs_m_a.loc[labs_m_a["turnaround_class"] == "poc",  "latency_min"]
core = labs_m_a.loc[labs_m_a["turnaround_class"] == "core", "latency_min"]
send = labs_m_a.loc[labs_m_a["turnaround_class"] == "send", "latency_min"]

print("\nInternal control (MIMIC-IV, median minutes):")
for nm, s in (("point of care / blood gas", poc), ("core laboratory", core), ("longer turnaround assays", send)):
    if len(s):
        print(f"  {nm:<32} {s.median():8,.0f}   n={len(s):,}")

if len(poc) and len(core) and poc.median() < core.median():
    ratio = core.median() / max(poc.median(), 1e-9)
    print(f"\n  Control satisfied: core laboratory results post {ratio:.1f}x slower than point of care.")
    print("  The interval is tracking laboratory processing rather than a clerical artefact.")
else:
    print("\n  Control NOT satisfied. Inspect the timestamp field before proceeding to Part 2.")

write_table(ctrl, "t7_latency_by_turnaround_class")

,turnaround_class,n,median (min),IQR (min),p90 (min),p99 (min)
1,poc,1418314,3.000,2-4,6.000,19.000
0,core,5070782,60.000,45-78,103.000,227.000
2,send,89251,72.000,50-112,217.000,720.000



Internal control (MIMIC-IV, median minutes):
  point of care / blood gas               3   n=1,418,314
  core laboratory                        60   n=5,070,782
  longer turnaround assays               72   n=89,251

  Control satisfied: core laboratory results post 20.0x slower than point of care.
  The interval is tracking laboratory processing rather than a clerical artefact.


WindowsPath('C:/Research_Paper_2/result_availability_audit/tables/t7_latency_by_turnaround_class.csv')

In [23]:
# ---- ordering priority: the strongest operational modifier ----------------
if labs_m_a["priority"].nunique() > 1:
    lat_pri = latency_summary(labs_m_a, ["priority", "turnaround_class"])
    lat_pri = lat_pri[lat_pri["n"] >= 500].sort_values(["turnaround_class", "p50"])
    display(fmt_iqr(lat_pri)[["priority", "turnaround_class", "n",
                              "median (min)", "IQR (min)", "p90 (min)", "p99 (min)"]])
    write_table(lat_pri, "t8_latency_by_priority")

    piv = (labs_m_a.groupby(["turnaround_class", "priority"])["latency_min"]
           .median().unstack())
    print("\nMedian latency in minutes, priority x turnaround class:")
    display(piv.reindex(CLASS_ORDER))
else:
    print("Priority field carries a single value in this extract; skipping.")

,priority,turnaround_class,n,median (min),IQR (min),p90 (min),p99 (min)
2,STAT,core,3491134,58.000,44-76,99.000,215.000
0,ROUTINE,core,1579648,62.000,48-82,110.000,248.000
5,UNSPECIFIED,poc,1418313,3.000,2-4,6.000,19.000
4,STAT,send,68516,68.000,47-106,196.000,671.900
1,ROUTINE,send,20735,84.000,62-136,288.000,865.000



Median latency in minutes, priority x turnaround class:


priority,ROUTINE,STAT,UNSPECIFIED
turnaround_class,,,
poc,NaN,7.000,3.000
core,62.000,58.000,NaN
send,84.000,68.000,NaN


In [24]:
# ---- circadian and weekly structure ---------------------------------------
lat_hour = latency_summary(labs_m_a, "draw_hour").sort_values("draw_hour")
lat_dow  = latency_summary(labs_m_a, "draw_dow").sort_values("draw_dow")
DOW = {0: "Mon", 1: "Tue", 2: "Wed", 3: "Thu", 4: "Fri", 5: "Sat", 6: "Sun"}
lat_dow["day"] = lat_dow["draw_dow"].map(DOW)

peak = lat_hour.loc[lat_hour["p50"].idxmax()]
trough = lat_hour.loc[lat_hour["p50"].idxmin()]
print(f"Slowest draw hour: {int(peak['draw_hour']):02d}:00  median {peak['p50']:,.0f} min")
print(f"Fastest draw hour: {int(trough['draw_hour']):02d}:00  median {trough['p50']:,.0f} min")
print(f"Within-day spread: {peak['p50'] - trough['p50']:,.0f} min "
      f"({peak['p50']/max(trough['p50'],1e-9):.2f}x)")

display(lat_dow[["day", "n", "p50", "p75", "p90"]])
write_table(lat_hour, "t9a_latency_by_draw_hour")
write_table(lat_dow,  "t9b_latency_by_draw_dow")

Slowest draw hour: 06:00  median 70 min
Fastest draw hour: 10:00  median 26 min
Within-day spread: 44 min (2.69x)


,day,n,p50,p75,p90
0,Mon,936691,52.000,72.000,96.000
1,Tue,941156,52.000,72.000,96.000
2,Wed,942674,52.000,72.000,97.000
3,Thu,941120,52.000,72.000,96.000
4,Fri,942487,52.000,72.000,96.000
5,Sat,937305,52.000,72.000,97.000
6,Sun,936914,52.000,72.000,96.000


WindowsPath('C:/Research_Paper_2/result_availability_audit/tables/t9b_latency_by_draw_dow.csv')

In [25]:
# ---- temporal stability across the collection era --------------------------
# MIMIC-IV dates are shifted per patient into a future window, so the calendar year on a
# timestamp carries no cross-patient meaning. `anchor_year_group` is the de-identified era
# label and is the only valid temporal stratifier in this database.
era = labs_m_a.merge(cohort_m[["stay_id", "anchor_year_group"]], on="stay_id", how="left")
lat_era = latency_summary(era, "anchor_year_group").sort_values("anchor_year_group")
lat_era = lat_era[lat_era["n"] >= 1000]

if len(lat_era):
    display(lat_era[["anchor_year_group", "n", "p50", "p75", "p90", "p99"]])
    write_table(lat_era, "t10_latency_by_era")
    if len(lat_era) > 1:
        first, last = lat_era.iloc[0], lat_era.iloc[-1]
        spread = lat_era["p50"].max() / max(lat_era["p50"].min(), 1e-9)
        print(f"\nMedian latency {first['anchor_year_group']}: {first['p50']:,.0f} min"
              f"  ->  {last['anchor_year_group']}: {last['p50']:,.0f} min")
        print(f"  Spread across eras: {spread:.2f}x")
        print("  A stable series supports treating latency as a structural property of laboratory")
        print("  operations rather than an artefact of a particular collection era.")
else:
    print("Insufficient era-stratified volume for a temporal stability check.")
    lat_era = pd.DataFrame()

del era; gc.collect()

,anchor_year_group,n,p50,p75,p90,p99
0,2008 - 2010,1932803,51.000,71.000,95.000,243.000
1,2011 - 2013,1530550,50.000,70.000,92.000,214.000
2,2014 - 2016,1583344,53.000,74.000,99.000,206.000
3,2017 - 2019,1531650,53.000,73.000,99.000,199.000



Median latency 2008 - 2010: 51 min  ->  2017 - 2019: 53 min
  Spread across eras: 1.06x
  A stable series supports treating latency as a structural property of laboratory
  operations rather than an artefact of a particular collection era.


0

In [26]:
# ---- care unit variation ---------------------------------------------------
cu = labs_m_a.merge(cohort_m[["stay_id", "first_careunit"]], on="stay_id", how="left")
lat_unit = latency_summary(cu, "first_careunit")
lat_unit = lat_unit[lat_unit["n"] >= 2000].sort_values("p50")
display(fmt_iqr(lat_unit)[["first_careunit", "n", "median (min)", "IQR (min)", "p90 (min)"]])
write_table(lat_unit, "t11_latency_by_careunit")
del cu; gc.collect()

,first_careunit,n,median (min),IQR (min),p90 (min)
0,Cardiac Vascular Intensive Care Unit (CVICU),1340086,32.000,3-60,81.000
8,Trauma SICU (TSICU),879244,49.000,22-68,89.000
7,Surgical Intensive Care Unit (SICU),1003944,54.000,31-72,95.000
2,Medical Intensive Care Unit (MICU),1305648,55.000,31-75,98.000
3,Medical/Surgical Intensive Care Unit (MICU/SICU),974724,56.000,36-76,102.000
1,Coronary Care Unit (CCU),719321,57.000,33-77,104.000
6,Neuro Surgical Intensive Care Unit (Neuro SICU),211885,59.000,41-81,111.000
4,Neuro Intermediate,100365,73.000,51-100,130.000
5,Neuro Stepdown,43130,78.000,55-110,144.000


0

In [27]:
# ---- eICU: between-hospital variation --------------------------------------
EICU_LATENCY_USABLE = bool((labs_e_a["latency_min"] > 0).mean() >= 0.05)

if EICU_LATENCY_USABLE:
    lat_hosp = latency_summary(labs_e_a, "hospitalid")
    lat_hosp = lat_hosp[lat_hosp["n"] >= 1000].sort_values("p50").reset_index(drop=True)

    hmeta = cohort_e[["hospitalid", "numbedscategory", "teachingstatus", "region"]].drop_duplicates("hospitalid")
    lat_hosp = lat_hosp.merge(hmeta, on="hospitalid", how="left")

    print(f"Hospitals with at least 1,000 analytic results: {len(lat_hosp)}")
    if len(lat_hosp) >= 5:
        print(f"Median latency across hospitals: {lat_hosp['p50'].min():,.0f} to {lat_hosp['p50'].max():,.0f} min")
        print(f"  IQR of hospital medians: {lat_hosp['p50'].quantile(.25):,.0f} to {lat_hosp['p50'].quantile(.75):,.0f} min")
        print(f"  Fold difference, slowest vs fastest: {lat_hosp['p50'].max()/max(lat_hosp['p50'].min(),1e-9):.1f}x")

    display(lat_hosp.head(15))
    write_table(lat_hosp, "t12_latency_by_hospital_eicu")

    for col in ("numbedscategory", "teachingstatus", "region"):
        if lat_hosp[col].notna().any():
            agg = lat_hosp.groupby(col)["p50"].agg(hospitals="size", median_of_medians="median").round(1)
            print(f"\nHospital median latency by {col}:")
            display(agg)
else:
    print("The eICU revised-result offset carries too little non-zero signal for a hospital-level")
    print("latency comparison. eICU is retained as a case-mix and ordering-intensity comparison site;")
    print("latency quantification is confined to MIMIC-IV. This is reported, not worked around.")
    lat_hosp = pd.DataFrame()

Hospitals with at least 1,000 analytic results: 190
Median latency across hospitals: 0 to 225 min
  IQR of hospital medians: 24 to 44 min
  Fold difference, slowest vs fastest: 225000000000.0x


,hospitalid,n,mean,p10,p25,p50,p75,p90,p95,p99,numbedscategory,teachingstatus,region
0,264,303341,12.700,0.000,0.000,0.000,0.000,51.000,76.000,153.000,>= 500,t,Midwest
1,249,11396,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,<100,f,Midwest
2,350,4814,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,None,f,None
3,345,37338,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,250 - 499,f,South
4,342,6961,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,<100,f,Midwest
5,337,23956,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,None,f,None
6,336,38311,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,100 - 249,f,Midwest
7,331,37493,0.100,0.000,0.000,0.000,0.000,0.000,0.000,0.000,None,f,None
8,328,11471,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,<100,f,Midwest
9,318,54968,0.100,0.000,0.000,0.000,0.000,0.000,0.000,0.000,250 - 499,f,South



Hospital median latency by numbedscategory:


,hospitals,median_of_medians
numbedscategory,,
100 - 249,60,39.000
250 - 499,35,35.000
<100,38,35.000
>= 500,23,38.000



Hospital median latency by teachingstatus:


,hospitals,median_of_medians
teachingstatus,,
f,172,37.000
t,18,34.500



Hospital median latency by region:


,hospitals,median_of_medians
region,,
Midwest,63,30.000
Northeast,13,0.000
South,52,40.000
West,40,39.000


In [28]:
# ---- is the between-hospital signal laboratory turnaround or contribution practice? -
hz = (labs_e_a.groupby("hospitalid")["latency_min"]
      .agg(n="size",
           pct_zero=lambda s: 100 * (s == 0).mean(),
           pct_neg=lambda s: 100 * (s < 0).mean(),
           p50="median"))
hz["all_zero"] = hz["p50"] == 0

n_zero = int(hz["all_zero"].sum())
rows_zero = int(hz.loc[hz["all_zero"], "n"].sum())
print(f"Hospitals with median latency of exactly zero: {n_zero} / {len(hz)}")
print(f"Results contributed by those hospitals: {rows_zero:,} "
      f"({100 * rows_zero / hz['n'].sum():.1f}% of all eICU results)")
display(hz["pct_zero"].describe())

fig, ax = plt.subplots(figsize=(6.6, 3.6))
ax.hist(hz["pct_zero"], bins=40, color="#2b6cb0", alpha=0.85)
ax.set_xlabel("Percentage of a hospital's results with zero latency")
ax.set_ylabel("Hospitals")
ax.set_title("Distribution of zero-latency rate across eICU hospitals")
save_fig(fig, "f8_eicu_zero_latency_distribution")
write_table(hz.reset_index(), "t18_eicu_hospital_zero_latency")

Hospitals with median latency of exactly zero: 43 / 206
Results contributed by those hospitals: 1,652,849 (18.8% of all eICU results)


count   206.000
mean     22.135
std      38.334
min       0.000
25%       0.000
50%       0.064
75%      15.774
max     100.000
Name: pct_zero, dtype: float64

WindowsPath('C:/Research_Paper_2/result_availability_audit/tables/t18_eicu_hospital_zero_latency.csv')

In [29]:
# ---- ordering intensity: a latency-independent cross-site comparison -------
# Sampling frequency determines how often a model's inputs can refresh at all, and it is
# measurable in both databases regardless of the eICU timestamp question.
int_m = (labs_m_a[labs_m_a["in_icu"]]
         .groupby("stay_id").size().rename("n_results").reset_index()
         .merge(cohort_m[["stay_id", "icu_hours"]], on="stay_id", how="left"))
int_m["results_per_24h"] = 24 * int_m["n_results"] / int_m["icu_hours"].clip(lower=1e-6)

int_e = (labs_e_a[labs_e_a["in_icu"]]
         .groupby("patientunitstayid").size().rename("n_results").reset_index()
         .merge(cohort_e[["patientunitstayid", "icu_hours", "hospitalid"]], on="patientunitstayid", how="left"))
int_e["results_per_24h"] = 24 * int_e["n_results"] / int_e["icu_hours"].clip(lower=1e-6)

intensity = pd.DataFrame([
    {"database": "MIMIC-IV", "stays_with_labs": len(int_m),
     "results per stay, median (IQR)": f"{int_m['n_results'].median():.0f} "
                                       f"({int_m['n_results'].quantile(.25):.0f}-{int_m['n_results'].quantile(.75):.0f})",
     "results per 24h, median (IQR)": f"{int_m['results_per_24h'].median():.1f} "
                                      f"({int_m['results_per_24h'].quantile(.25):.1f}-{int_m['results_per_24h'].quantile(.75):.1f})"},
    {"database": "eICU-CRD", "stays_with_labs": len(int_e),
     "results per stay, median (IQR)": f"{int_e['n_results'].median():.0f} "
                                       f"({int_e['n_results'].quantile(.25):.0f}-{int_e['n_results'].quantile(.75):.0f})",
     "results per 24h, median (IQR)": f"{int_e['results_per_24h'].median():.1f} "
                                      f"({int_e['results_per_24h'].quantile(.25):.1f}-{int_e['results_per_24h'].quantile(.75):.1f})"},
]).set_index("database")
display(intensity)
write_table(intensity, "t13_ordering_intensity", index=True)

,stays_with_labs,"results per stay, median (IQR)","results per 24h, median (IQR)"
database,,,
MIMIC-IV,48214,72 (38-131),33.7 (24.0-48.1)
eICU-CRD,112468,38 (19-73),20.4 (14.8-28.1)


WindowsPath('C:/Research_Paper_2/result_availability_audit/tables/t13_ordering_intensity.csv')

In [30]:
# ---- microbiology: the long tail -------------------------------------------
mic_spec = latency_summary(micro_a, "specimen")
mic_spec = mic_spec[mic_spec["n"] >= 500].sort_values("p50")
mic_spec[["p50_h", "p90_h", "p99_h"]] = (mic_spec[["p50", "p90", "p99"]] / 60).round(1)
display(mic_spec[["specimen", "n", "p50_h", "p90_h", "p99_h"]].rename(
    columns={"p50_h": "median (h)", "p90_h": "p90 (h)", "p99_h": "p99 (h)"}))
write_table(mic_spec, "t14_microbiology_latency_by_specimen")

pos = micro_a[micro_a["organism_isolated"]]
if len(pos) > 100:
    print(f"\nCultures with an organism isolated: {len(pos):,}")
    print(f"  Median time to availability: {pos['latency_h'].median():.1f} h "
          f"(IQR {pos['latency_h'].quantile(.25):.1f}-{pos['latency_h'].quantile(.75):.1f})")
    print(f"  p90: {pos['latency_h'].quantile(.90):.1f} h")
    print("\n  A model using a culture feature at a scoring time earlier than this is using a")
    print("  result that did not exist in the record at that moment.")

,specimen,n,median (h),p90 (h),p99 (h)
39,Influenza A/B by DFA,1123,8.800,25.300,83.500
57,STOOL,9796,19.700,57.100,132.600
54,SEROLOGY/BLOOD,2368,30.000,82.000,166.500
52,Rapid Respiratory Viral Screen & Culture,3913,38.800,97.100,141.100
69,URINE,59690,42.200,85.600,140.600
37,IMMUNOLOGY,743,49.200,99.300,134.800
14,BRONCHIAL WASHINGS,2332,49.800,155.100,332.000
16,Blood (CMV AB),907,50.400,90.400,116.100
21,CATHETER TIP-IV,2793,50.600,91.200,189.400
15,BRONCHOALVEOLAR LAVAGE,12245,51.000,145.100,330.900



Cultures with an organism isolated: 112,281
  Median time to availability: 76.2 h (IQR 57.9-106.5)
  p90: 146.1 h

  A model using a culture feature at a scoring time earlier than this is using a
  result that did not exist in the record at that moment.


---
## 11. Visibility curves

The descriptive tables above describe results. This section describes the *record*, which is what a
deployed model actually queries.

For a model scoring a patient at hour `T` after ICU admission, the question is what proportion of the
laboratory results already drawn by `T` are visible at `T`. Anything drawn but not yet released is
information the model is credited with during development and denied during deployment.

Two quantities are computed:

* **Result visibility** — of all results drawn in the first `T` hours, the proportion released by `T`.
* **Stay-level completeness** — for each analyte, the proportion of stays whose most recent value at
  `T` on the observation clock is the same value the availability clock would return. Where these
  differ, the model is scoring on a different number than it would in deployment.

In [31]:
with Stage("visibility-curve"):
    lm = labs_m_a[labs_m_a["hours_from_icu_admit"] >= 0][
        ["stay_id", "analyte", "turnaround_class", "hours_from_icu_admit", "latency_h"]].copy()
    lm["avail_h"] = lm["hours_from_icu_admit"] + lm["latency_h"]

    rows = []
    for T in CFG.visibility_hours:
        drawn = lm["hours_from_icu_admit"] <= T
        n_drawn = int(drawn.sum())
        if n_drawn == 0:
            continue
        vis = drawn & (lm["avail_h"] <= T)
        rows.append({"scoring_hour": T, "n_drawn": n_drawn, "n_visible": int(vis.sum()),
                     "visible_pct": round(100 * vis.sum() / n_drawn, 2), "stratum": "all"})
        for cls in CLASS_ORDER:
            sel = lm["turnaround_class"] == cls
            d = drawn & sel
            if d.sum() < 100:
                continue
            v = d & (lm["avail_h"] <= T)
            rows.append({"scoring_hour": T, "n_drawn": int(d.sum()), "n_visible": int(v.sum()),
                         "visible_pct": round(100 * v.sum() / d.sum(), 2), "stratum": cls})

    vis_curve = pd.DataFrame(rows)

write_table(vis_curve, "t15_visibility_curve")

allc = vis_curve[vis_curve["stratum"] == "all"].set_index("scoring_hour")["visible_pct"]
for T in (3, 6, 12, 24, 48):
    if T in allc.index:
        print(f"  At hour {T:>2}: {allc.loc[T]:5.1f}% of drawn results are visible "
              f"({100-allc.loc[T]:.1f}% invisible to a model scoring at that moment)")

[visibility-curve] start
[visibility-curve] done in 2.16 min
  At hour  3:  79.4% of drawn results are visible (20.6% invisible to a model scoring at that moment)
  At hour  6:  91.0% of drawn results are visible (9.0% invisible to a model scoring at that moment)
  At hour 12:  95.0% of drawn results are visible (5.0% invisible to a model scoring at that moment)
  At hour 24:  98.2% of drawn results are visible (1.8% invisible to a model scoring at that moment)
  At hour 48:  99.3% of drawn results are visible (0.7% invisible to a model scoring at that moment)


In [32]:
# ---- stay-level: how often does the two-clock choice change the value used? -
with Stage("value-substitution"):
    SUB_HOURS = [6, 12, 24, 48]
    key_analytes = [a for a in ["creatinine", "lactate", "wbc", "platelet", "bicarbonate",
                                "urea_nitrogen", "hemoglobin", "potassium"]
                    if a in set(lm["analyte"])]
    lmv = labs_m_a[(labs_m_a["hours_from_icu_admit"] >= 0) &
                   (labs_m_a["analyte"].isin(key_analytes))][
        ["stay_id", "analyte", "hours_from_icu_admit", "latency_h", "valuenum"]].copy()
    lmv["avail_h"] = lmv["hours_from_icu_admit"] + lmv["latency_h"]
    lmv = lmv.dropna(subset=["valuenum"])

    out = []
    for T in SUB_HOURS:
        obs = (lmv[lmv["hours_from_icu_admit"] <= T]
               .sort_values("hours_from_icu_admit")
               .groupby(["stay_id", "analyte"], as_index=False).last()
               .rename(columns={"valuenum": "value_obs_clock"}))
        av = (lmv[lmv["avail_h"] <= T]
              .sort_values("avail_h")
              .groupby(["stay_id", "analyte"], as_index=False).last()
              .rename(columns={"valuenum": "value_avail_clock"}))
        j = obs[["stay_id", "analyte", "value_obs_clock"]].merge(
            av[["stay_id", "analyte", "value_avail_clock"]],
            on=["stay_id", "analyte"], how="left")
        j["absent_on_avail_clock"] = j["value_avail_clock"].isna()
        j["value_differs"] = (~j["absent_on_avail_clock"]) & (
            (j["value_obs_clock"] - j["value_avail_clock"]).abs() > 1e-9)
        g = j.groupby("analyte").agg(
            stays=("stay_id", "size"),
            absent_pct=("absent_on_avail_clock", lambda s: round(100*s.mean(), 2)),
            differs_pct=("value_differs", lambda s: round(100*s.mean(), 2))).reset_index()
        g["scoring_hour"] = T
        g["discordant_pct"] = (g["absent_pct"] + g["differs_pct"]).round(2)
        out.append(g)

    subst = pd.concat(out, ignore_index=True)[
        ["scoring_hour", "analyte", "stays", "absent_pct", "differs_pct", "discordant_pct"]]

display(subst.pivot(index="analyte", columns="scoring_hour", values="discordant_pct"))
write_table(subst, "t16_value_substitution")

print("\nEach cell is the percentage of stays where the value a model would use differs between the")
print("two clocks at that scoring hour, either because the result was not yet released or because a")
print("different result was the most recent one available.")
del lmv, lm; gc.collect()

[value-substitution] start
[value-substitution] done in 16.9 s


scoring_hour,6,12,24,48
analyte,,,,
bicarbonate,14.990,9.460,4.610,2.580
creatinine,14.310,8.700,4.110,2.230
hemoglobin,10.400,6.980,3.430,1.880
lactate,1.240,0.750,0.380,0.210
platelet,9.760,6.550,3.030,1.580
potassium,16.460,11.560,5.640,3.130
urea_nitrogen,14.730,9.580,4.760,2.660
wbc,9.290,6.370,2.890,1.530



Each cell is the percentage of stays where the value a model would use differs between the
two clocks at that scoring hour, either because the result was not yet released or because a
different result was the most recent one available.


9

---
## 12. Figures

In [33]:
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 200, "font.size": 10,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.25, "grid.linewidth": 0.6,
})
CLS_COLOR = {"poc": "#2b6cb0", "core": "#c05621", "send": "#276749"}
CLS_LABEL = {"poc": "Point of care / blood gas", "core": "Core laboratory", "send": "Longer turnaround"}

# --- F1: empirical CDF of latency by turnaround class ----------------------
fig, ax = plt.subplots(figsize=(7.2, 4.4))
for cls in CLASS_ORDER:
    s = labs_m_a.loc[labs_m_a["turnaround_class"] == cls, "latency_min"]
    if len(s) < 100:
        continue
    s = s.sample(min(len(s), 400_000), random_state=CFG.seed).sort_values().values
    ax.plot(s, np.arange(1, len(s)+1)/len(s)*100, lw=2,
            color=CLS_COLOR[cls], label=f"{CLS_LABEL[cls]}  (n={len(s):,})")
for x in (60, 180, 360):
    ax.axvline(x, color="grey", ls=":", lw=0.9)
    ax.text(x, 3, f"{x//60}h", ha="center", fontsize=8, color="grey")
ax.set_xscale("symlog", linthresh=10)
ax.set_xlim(0, CFG.latency_max_minutes)
ax.set_ylim(0, 100)
ax.set_xlabel("Minutes from observation to availability (symlog scale)")
ax.set_ylabel("Cumulative percentage of results")
ax.set_title("Result availability latency by turnaround class, MIMIC-IV")
ax.legend(frameon=False, loc="lower right", fontsize=9)
save_fig(fig, "f1_latency_ecdf_by_class")

WindowsPath('C:/Research_Paper_2/result_availability_audit/figures/f1_latency_ecdf_by_class.png')

In [34]:
# --- F2: latency by analyte, ordered ----------------------------------------
d = lat_analyte[(lat_analyte["db"] == "MIMIC-IV") & (lat_analyte["n"] >= 500)].sort_values("p50")
fig, ax = plt.subplots(figsize=(7.6, max(4.5, 0.28*len(d))))
y = np.arange(len(d))
cols = [CLS_COLOR[c] for c in d["turnaround_class"]]
ax.hlines(y, d["p25"], d["p75"], color=cols, lw=5, alpha=0.55)
ax.hlines(y, d["p75"], d["p90"], color=cols, lw=1.8, alpha=0.5)
ax.scatter(d["p50"], y, color=cols, s=34, zorder=3)
ax.set_yticks(y)
ax.set_yticklabels([a.replace("_", " ") for a in d["analyte"]])
ax.set_xscale("log")
ax.set_xlabel("Minutes from observation to availability (log scale)")
ax.set_title("Result availability latency by analyte, MIMIC-IV\npoint = median, bar = IQR, whisker to p90")
handles = [plt.Line2D([], [], color=CLS_COLOR[c], lw=5, label=CLS_LABEL[c]) for c in CLASS_ORDER]
ax.legend(handles=handles, frameon=False, fontsize=9, loc="lower right")
save_fig(fig, "f2_latency_by_analyte")

WindowsPath('C:/Research_Paper_2/result_availability_audit/figures/f2_latency_by_analyte.png')

In [35]:
# --- F3: circadian structure -------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(11, 3.9))
a = axes[0]
a.plot(lat_hour["draw_hour"], lat_hour["p50"], "o-", color="#2b6cb0", lw=2, ms=4, label="median")
a.fill_between(lat_hour["draw_hour"], lat_hour["p25"], lat_hour["p75"], color="#2b6cb0", alpha=0.18, label="IQR")
a.plot(lat_hour["draw_hour"], lat_hour["p90"], "--", color="#c05621", lw=1.4, label="p90")
a.set_xticks(range(0, 24, 3))
a.set_xlabel("Hour of day at observation")
a.set_ylabel("Latency (minutes)")
a.set_title("Latency by hour of observation")
a.legend(frameon=False, fontsize=8)

b = axes[1]
b.bar(lat_dow["day"], lat_dow["p50"], color="#2b6cb0", alpha=0.8)
b.errorbar(lat_dow["day"], lat_dow["p50"],
           yerr=[lat_dow["p50"]-lat_dow["p25"], lat_dow["p75"]-lat_dow["p50"]],
           fmt="none", ecolor="#4a5568", capsize=3, lw=1)
b.set_ylabel("Median latency (minutes)")
b.set_title("Latency by day of week")
fig.tight_layout()
save_fig(fig, "f3_latency_circadian")

WindowsPath('C:/Research_Paper_2/result_availability_audit/figures/f3_latency_circadian.png')

In [36]:
# --- F4: visibility curve ----------------------------------------------------
fig, ax = plt.subplots(figsize=(7.2, 4.4))
a_all = vis_curve[vis_curve["stratum"] == "all"]
ax.plot(a_all["scoring_hour"], a_all["visible_pct"], lw=2.6, color="#1a202c", label="All analytes")
for cls in CLASS_ORDER:
    s = vis_curve[vis_curve["stratum"] == cls]
    if len(s):
        ax.plot(s["scoring_hour"], s["visible_pct"], lw=1.8, color=CLS_COLOR[cls],
                ls="--", label=CLS_LABEL[cls])
ax.axhline(100, color="grey", lw=0.8, ls=":")
ax.set_xlabel("Scoring hour after ICU admission")
ax.set_ylabel("Percentage of drawn results visible at scoring time")
ax.set_title("Record visibility at the moment of scoring, MIMIC-IV")
ax.set_xlim(min(CFG.visibility_hours), max(CFG.visibility_hours))
ax.legend(frameon=False, fontsize=9, loc="lower right")
save_fig(fig, "f4_visibility_curve")

WindowsPath('C:/Research_Paper_2/result_availability_audit/figures/f4_visibility_curve.png')

In [37]:
# --- F5: value substitution heatmap ------------------------------------------
piv = subst.pivot(index="analyte", columns="scoring_hour", values="discordant_pct")
piv = piv.sort_values(by=max(SUB_HOURS), ascending=False)
fig, ax = plt.subplots(figsize=(6.4, max(3.2, 0.42*len(piv))))
im = ax.imshow(piv.values, aspect="auto", cmap="OrRd", vmin=0)
ax.set_xticks(range(piv.shape[1]))
ax.set_xticklabels([f"h{c}" for c in piv.columns])
ax.set_yticks(range(piv.shape[0]))
ax.set_yticklabels([a.replace("_", " ") for a in piv.index])
for i in range(piv.shape[0]):
    for j in range(piv.shape[1]):
        v = piv.values[i, j]
        if np.isfinite(v):
            ax.text(j, i, f"{v:.0f}", ha="center", va="center", fontsize=8,
                    color="white" if v > np.nanmax(piv.values)*0.6 else "#1a202c")
ax.set_title("Percentage of stays where the value used differs between clocks")
ax.grid(False)
fig.colorbar(im, ax=ax, fraction=0.035, pad=0.03, label="% of stays")
save_fig(fig, "f5_value_substitution")

WindowsPath('C:/Research_Paper_2/result_availability_audit/figures/f5_value_substitution.png')

In [38]:
# --- F6: microbiology latency -------------------------------------------------
d = mic_spec.sort_values("p50").tail(14)
fig, ax = plt.subplots(figsize=(7.4, max(3.6, 0.34*len(d))))
y = np.arange(len(d))
ax.hlines(y, d["p25"]/60, d["p75"]/60, color="#553c9a", lw=5, alpha=0.55)
ax.hlines(y, d["p75"]/60, d["p90"]/60, color="#553c9a", lw=1.8, alpha=0.5)
ax.scatter(d["p50"]/60, y, color="#553c9a", s=34, zorder=3)
ax.set_yticks(y); ax.set_yticklabels(d["specimen"], fontsize=8)
ax.set_xlabel("Hours from collection to availability")
ax.set_title("Microbiology result availability latency by specimen, MIMIC-IV\npoint = median, bar = IQR, whisker to p90")
save_fig(fig, "f6_microbiology_latency")

WindowsPath('C:/Research_Paper_2/result_availability_audit/figures/f6_microbiology_latency.png')

In [39]:
# --- F7: eICU between-hospital variation (only when the proxy carries signal) --
if len(lat_hosp) >= 10:
    d = lat_hosp.sort_values("p50").reset_index(drop=True)
    fig, ax = plt.subplots(figsize=(7.6, 4.2))
    x = np.arange(len(d))
    ax.vlines(x, d["p25"], d["p75"], color="#2b6cb0", lw=1.2, alpha=0.5)
    ax.scatter(x, d["p50"], s=16, color="#1a202c", zorder=3)
    ax.axhline(d["p50"].median(), color="#c05621", ls="--", lw=1.4,
               label=f"median of hospital medians = {d['p50'].median():,.0f} min")
    ax.set_xlabel("Hospital, ordered by median latency")
    ax.set_ylabel("Latency (minutes)")
    ax.set_title(f"Between-hospital variation in result availability latency, eICU-CRD (n={len(d)} hospitals)")
    ax.legend(frameon=False, fontsize=9)
    save_fig(fig, "f7_hospital_variation_eicu")
elif EICU_LATENCY_USABLE:
    print(f"Only {len(lat_hosp)} hospitals cleared the volume threshold; the caterpillar plot")
    print("needs at least 10 to be informative. The underlying table is still written.")
else:
    print("Skipped: the eICU revised-result offset carried no usable latency signal (section 10).")

---
## 13. Provenance record

Every source file, cached artefact, table, and figure is hashed and chained. The terminal digest
fixes the entire run.

In [40]:
for nm, df in [("cohort_mimic", cohort_m), ("cohort_eicu", cohort_e)]:
    p = DIR["cache"] / f"{nm}.parquet"
    if not any(r["name"] == nm for r in PROV.records):
        PROV.add("cache", nm, p, {"rows": int(len(df))})

manifest = PROV.manifest()
manifest["part1_summary"] = {
    "mimic_stays": int(len(cohort_m)),
    "eicu_stays": int(len(cohort_e)),
    "eicu_hospitals": int(cohort_e["hospitalid"].nunique()),
    "mimic_lab_results_analytic": int(len(labs_m_a)),
    "eicu_lab_results_analytic": int(len(labs_e_a)),
    "microbiology_results_analytic": int(len(micro_a)),
    "analytes_resolved_mimic": int(labs_m_a["analyte"].nunique()),
    "analytes_resolved_eicu": int(labs_e_a["analyte"].nunique()),
    "analytes_in_both": SHARED,
    "median_latency_min_mimic": float(labs_m_a["latency_min"].median()),
    "p90_latency_min_mimic": float(labs_m_a["latency_min"].quantile(0.90)),
    "median_latency_min_by_class": {
        c: float(labs_m_a.loc[labs_m_a["turnaround_class"] == c, "latency_min"].median())
        for c in CLASS_ORDER if (labs_m_a["turnaround_class"] == c).any()
    },
    "eicu_nonzero_latency_fraction": float((labs_e_a["latency_min"] > 0).mean()),
    "visible_pct_at_hour": {int(h): float(allc.loc[h]) for h in (6, 12, 24, 48) if h in allc.index},
}

MANIFEST_P = OUT / "part1_manifest.json"
with open(MANIFEST_P, "w") as f:
    json.dump(manifest, f, indent=2, default=str)

print(f"Manifest written: {MANIFEST_P}")
print(f"Terminal chain digest: {PROV.chain}")
print(f"\nArtefacts: {sum(1 for r in PROV.records if r['role']=='table')} tables, "
      f"{sum(1 for r in PROV.records if r['role']=='figure')} figures, "
      f"{sum(1 for r in PROV.records if r['role']=='cache')} cached datasets")

t = pd.DataFrame([{"stage": k, "seconds": float(v["seconds"])}
                  for k, v in PROV.timings.items()])
if len(t):
    t["minutes"] = (t["seconds"] / 60).round(2)
    display(t.sort_values("seconds", ascending=False).reset_index(drop=True))

Manifest written: C:\Research_Paper_2\result_availability_audit\part1_manifest.json
Terminal chain digest: c33a334d6c4d9b9a3c109f96753b8ccf3ecc5325a1566341cad414a753eaddd0

Artefacts: 18 tables, 8 figures, 2 cached datasets


,stage,seconds,minutes
0,visibility-curve,129.430,2.160
1,eicu-labname-scan,48.930,0.820
2,value-substitution,16.920,0.280
3,labs-eicu,6.170,0.100
4,labs-mimic,3.590,0.060
5,source-fingerprints,1.960,0.030
6,cohort-eicu,0.290,0.000
7,micro-mimic,0.290,0.000
8,cohort-mimic,0.100,0.000


In [41]:
# ---- checks that must pass before Part 2 -----------------------------------
checks = []

med_poc  = labs_m_a.loc[labs_m_a["turnaround_class"] == "poc",  "latency_min"].median()
med_core = labs_m_a.loc[labs_m_a["turnaround_class"] == "core", "latency_min"].median()

checks.append(("Latency is measurable in MIMIC-IV",
               bool(labs_m_a["latency_min"].notna().mean() > 0.95),
               f"{100*labs_m_a['latency_min'].notna().mean():.1f}% of rows have a valid interval"))

checks.append(("Median latency is operationally meaningful (>15 min)",
               bool(labs_m_a["latency_min"].median() > 15),
               f"median {labs_m_a['latency_min'].median():.0f} min"))

checks.append(("Internal control holds (core slower than point of care)",
               bool(med_core > med_poc),
               f"core {med_core:.0f} min vs point of care {med_poc:.0f} min"))

checks.append(("Tail is long enough to displace hourly scoring (p90 > 60 min)",
               bool(labs_m_a["latency_min"].quantile(0.90) > 60),
               f"p90 {labs_m_a['latency_min'].quantile(0.90):.0f} min"))

# Deterioration alerts fire in the early hours of a stay, so hour 6 is the operationally
# relevant anchor. Hour 24 is reported alongside it for context.
vis6  = allc.loc[6]  if 6  in allc.index else np.nan
vis24 = allc.loc[24] if 24 in allc.index else np.nan
checks.append(("Record incompleteness in the early window is material (>5% invisible at h6)",
               bool(np.isfinite(vis6) and (100 - vis6) > 5),
               f"{100-vis6:.1f}% invisible at hour 6; {100-vis24:.1f}% at hour 24"))

checks.append(("Timestamp inconsistencies are rare (<2% negative)",
               bool((labs_m["latency_min"] < 0).mean() < 0.02),
               f"{100*(labs_m['latency_min'] < 0).mean():.2f}% negative"))

checks.append(("eICU carries usable latency signal",
               bool((labs_e_a["latency_min"] > 0).mean() >= 0.05),
               f"{100*(labs_e_a['latency_min'] > 0).mean():.1f}% non-zero (informational, not blocking)"))

res = pd.DataFrame(checks, columns=["check", "passed", "observed"])
res["status"] = np.where(res["passed"], "PASS", "REVIEW")
display(res[["check", "status", "observed"]])
write_table(res, "t17_gate_checks")

blocking = res.iloc[:6]
if blocking["passed"].all():
    print("\nAll blocking checks passed. The latency variable behaves as a laboratory processing")
    print("interval and the record is materially incomplete at realistic scoring times.")
    print("Proceed to Part 2: escalation events, two-clock hourly feature matrices, model scoring.")
else:
    print("\nOne or more blocking checks needs  review:")
    for _, r in blocking[~blocking["passed"]].iterrows():
        print(f"  - {r['check']}: {r['observed']}")
    print("\nInspect before building Part 2 on this variable.")

,check,status,observed
0,Latency is measurable in MIMIC-IV,PASS,100.0% of rows have a valid interval
1,Median latency is operationally meaningful (>1...,PASS,median 52 min
2,Internal control holds (core slower than point...,PASS,core 60 min vs point of care 3 min
3,Tail is long enough to displace hourly scoring...,PASS,p90 96 min
4,Record incompleteness in the early window is m...,PASS,9.0% invisible at hour 6; 1.8% at hour 24
5,Timestamp inconsistencies are rare (<2% negative),PASS,0.00% negative
6,eICU carries usable latency signal,PASS,"78.4% non-zero (informational, not blocking)"



All blocking checks passed. The latency variable behaves as a laboratory processing
interval and the record is materially incomplete at realistic scoring times.
Proceed to Part 2: escalation events, two-clock hourly feature matrices, model scoring.


In [42]:
if CON is not None:
    CON.close()
print("Part 1 complete.")
print(f"Outputs under: {OUT}")
for k, d in DIR.items():
    n = len(list(d.glob('*'))) if d.exists() else 0
    print(f"  {k:<8} {n:>3} files   {d}")

Part 1 complete.
Outputs under: C:\Research_Paper_2\result_availability_audit
  cache      5 files   C:\Research_Paper_2\result_availability_audit\cache
  tables    21 files   C:\Research_Paper_2\result_availability_audit\tables
  figs       8 files   C:\Research_Paper_2\result_availability_audit\figures
  logs       0 files   C:\Research_Paper_2\result_availability_audit\logs
